In [ ]:
import json
from collections import Counter
from datetime import datetime, timedelta
from itertools import product, zip_longest
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as goa
import regex
import requests
from plotly.colors import qualitative, sample_colorscale
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

from datetime import datetime
import pytz
from src.utils import (
    guardarExcel,
    guardarExcelMulti
)

from datetime import timedelta
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.api import getHistoricoMOW
from src.api.APIs import getInfoToposMSE, getInfoToposMSEWithoutCTC
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
from src.utils import (
    isEmpty,
    loadEstaciones,
    loadLocalizaciones,
    localizeFecha,
    parallelizeFunction,
    rellenarId,
    removeDoubleQuotes,
    splitDataframe,
    getEstacionamientos,
    loadEstacionSinCTC
    
)
from src.api.api import GraylogAPIProcessor
from src.utils.util import loadEstaciones,loadEstacionComercial
from src.api.APIs import getCirculacionesPlanificadas,getCirculacionesComerciales
from src.utils.topos import getEstacionamientos

In [ ]:
from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime
)
from src.processor import SitraProcessor, MIEProcessor, XPECProcessor


In [ ]:
estaciones = getEstacionamientos()

In [ ]:
est= loadEstaciones()

In [ ]:
mac = est[est["CTC"] == "MAC"]

In [ ]:
mac[mac["Código"] == "70107"]

In [ ]:
mac

In [ ]:
estaciones_mac = estaciones[estaciones["Código"].isin(mac["Código"])]

In [ ]:
estaciones_mac[estaciones_mac["Código"] == "70107"]


In [ ]:
estaciones_mac_1=estaciones_mac.merge(
    mac[["Código","Mnemónico"]],
    on = "Código",
    how = "left"
)

In [ ]:
estaciones_mac_1[estaciones_mac_1["Código"] == "70107"]

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\OneDrive - Ingeniería y Economía del Transporte S.A\Backlog\Data\Informe_puntual\estacionamiento_mac_2.xlsx")

In [ ]:
guardarExcel(estaciones_mac_1,fname)

In [ ]:
# Tipos de tren que queremos
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "SUPRESIÓN",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ORIGEN",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    "Stopped": "STOP",
    "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "MANIOBRA_APROXIMACION"
            "MANIOBRA_LLEGADA",
            "MANIOBRA_SALIDA",
        ]
    )
}

In [ ]:

def cargarHistorico(
    start_date: str,
    end_date: str,
    estaciones: list[str],
    trenes: list[str],
    xSIV: bool = True,
    xSIVPLUS:bool = False,
    jCTC: bool = False,
    xREG: bool = False,
    pro: bool = True,
    maniobra:bool = True
):
    # Comprobamos que la fecha de fin sea después de la de inicio
    if end_date <= start_date:
        end_date = (pd.to_datetime(start_date) + timedelta(days=1)).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    historico = getHistoricoMOW(
        estaciones=estaciones,
        trenes=trenes,
        inicio=start_date,
        fin=end_date,
        xSIV=xSIV,
        xSIVPLUS = xSIVPLUS,
        jCTC=jCTC,
        xREG=xREG,
        pro=pro,
        maniobra= maniobra
    )
        
    if (xREG == False): 
        historico = historico[
            (historico["Fecha"] >= pd.to_datetime(start_date))
            & (historico["Fecha"] <= pd.to_datetime(end_date))
        ]
    else:
         historico = historico[
            (historico["FechaHora"] >= pd.to_datetime(start_date))
            & (historico["FechaHora"] <= pd.to_datetime(end_date))
        ]
    # Usamos movimientos auditados
    # historico = historico[
    #     np.invert(historico["FuenteVía"].isin(["PLANNED", "SITRA_PROVIDED"]))
    # ]
    
    
    
    if (xREG == False): 
        historico = historico[historico["NTécnico"].apply(isValidCode)].dropna(
            subset=["Movimiento"]
        )
        historico["mov_ord"] = historico["Movimiento"].apply(mov_sorter.get)
    return historico


In [ ]:
start_date = "2026-05-25"
end_date = "2026-05-27"
estaciones = []

In [ ]:

estaciones = []
ntrenes = [rellenarId(el) for el in np.arange(100000)]
# ntrenes = [rellenarId(el) for el in np.arange(2000, 6000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=True,
    xSIVPLUS=False,
    jCTC=False,
    pro=True,
)
# historico_pro = historico_pro.sort_values(
#     by=["FechaOrigen", "NTécnico", "Fecha"]
# ).reset_index(drop=True)

# # Añadir información de la fecha
# historico_pro["Día"] = historico_pro["Fecha"].dt.date
# historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
# historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
# historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
df = historico_pro.copy()

In [ ]:
df_cercania = df[df["Producto"].isin(["CERCANIA","CERCANIAS RAM"])].copy()

In [ ]:
df_maniobra = df_cercania[df_cercania["Movimiento"].isin(["MANIOBRA_SALIDA","MANIOBRA_LLEGADA"])].copy()

In [ ]:
df_maniobra_filter =df_maniobra[["Nombre","Código"]].copy()

In [ ]:
df_maniobra_filter.drop_duplicates(inplace=True)

In [ ]:
df_maniobra_filter


In [ ]:
fname= Path(r"C:\Users\xiangzhou.zhang\OneDrive - Ingeniería y Economía del Transporte S.A\Backlog\Data\Informe_puntual\Estacion_maniobra.xlsx")

In [ ]:
guardarExcel(df_maniobra_filter,fname)

In [ ]:
salidas = df_filter[df_filter["Movimiento"] == "SALIDA"].copy()

In [ ]:
salidas.head(4)

In [ ]:
resultado = salidas.merge(
    estacionamientos[['Código', 'VíaTécnica','Vía']],
    left_on=['Código','Vía', 'Elemento',],
    right_on=['Código','Vía','VíaTécnica'],
    how='left',
    indicator=True
)


In [ ]:
resultado['coincide'] = resultado['_merge'] == 'both'
resultado.drop(columns=['_merge', 'VíaTécnica'], inplace=True)

In [ ]:
df_resultado = resultado[["NTécnico", "Código","Nombre","Elemento","Vía","coincide"]]

In [ ]:
no_coincide = df_resultado[df_resultado["coincide"] == False].drop(columns ="coincide")

In [ ]:

estaciones = []
ntrenes = [rellenarId(el) for el in np.arange(100000)]
# ntrenes = [rellenarId(el) for el in np.arange(2000, 6000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=False,
    xSIVPLUS=False,
    jCTC=True,
    pro=True,
)
# historico_pro = historico_pro.sort_values(
#     by=["FechaOrigen", "NTécnico", "Fecha"]
# ).reset_index(drop=True)

# # Añadir información de la fecha
# historico_pro["Día"] = historico_pro["Fecha"].dt.date
# historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
# historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
# historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
no_coincide

In [ ]:
estacionamientos

In [ ]:
no_coincide_1 = no_coincide.merge(
    estacionamientos[["Código","VíaTécnica","Vía"]],
    on = ["Código","Vía"],
    how = "left"

)

In [ ]:
no_coincide_1

In [ ]:
JCTC= historico_pro[["Fecha","NTécnico","Código","Elemento","Movimiento"]].copy()

In [ ]:
JCTC.rename(columns= {"Elemento":"VíaTécnica"}, inplace= True)


In [ ]:
JCTC

In [ ]:
JCTC_OCUPA = JCTC[JCTC["Movimiento"] == "LLEGADA"].copy()

In [ ]:
JCTC_LIBERA = JCTC[JCTC["Movimiento"] == "SALIDA"].copy()

In [ ]:
no_coincide_2 = no_coincide_1.merge(
    JCTC_OCUPA,
    on = ["NTécnico","Código","VíaTécnica"],
    how = "left"

)

In [ ]:
no_coincide_2

In [ ]:
ocupas = no_coincide_2.drop_duplicates(subset=["NTécnico","Código","VíaTécnica"]).copy()

In [ ]:
ocupas

In [ ]:
JCTC_LIBERA.rename(columns={"VíaTécnica":"Elemento","Fecha":"Fecha_Salida","Movimiento":"Movimiento_Libera"}, inplace=True)

In [ ]:
no_coincide_3 = ocupas.merge(

    JCTC_LIBERA,
    on = ["NTécnico","Código","Elemento"],
    how = "left"
)

In [ ]:
no_coincide_3

In [ ]:
no_coincide_3.rename(columns={"Fecha":"Fecha_ocupa_estacionamiento","Fecha_Salida":"Fecha_libera_CV_No_estacionamiento"}, inplace=True)

In [ ]:
no_coincide_3.drop(columns=["Movimiento","Movimiento_Libera"])

In [ ]:
no_coincide_3 = no_coincide_3[["NTécnico","Código","Nombre","Elemento","Vía","VíaTécnica","Fecha_libera_CV_No_estacionamiento","Fecha_ocupa_estacionamiento"]].copy()

In [ ]:
no_coincide_3.rename(columns={"Elemento":"CV de libera","VíaTécnica":"CV Estacionaiento"},inplace=True)

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\OneDrive - Ingeniería y Economía del Transporte S.A\Backlog\Data\Informe_puntual\vía_no_estacionmiento_5.xlsx")

In [ ]:
no_coincide_2

In [ ]:
guardarExcel(no_coincide_3,fname)

In [ ]:
estacionamientos[estacionamientos["Código"]=="37705"]

In [ ]:
df_filter = df[(df["Movimiento"] == "LLEGADA") | (df["Movimiento"] == "IN_FORECASTED")].copy()

In [ ]:
df_filter = df_filter[df_filter["FuenteMovimiento"] != "SITRA"].copy()


In [ ]:
sub_dfs = [grupo for _, grupo in df_filter.groupby(["NTécnico", "Código"])]

In [ ]:
sub_dfs[0].columns

In [ ]:
from datetime import timedelta

posteriores = []
no_posteriores = []
sin_datos = []

for df in sub_dfs:
    llegada_rows = df[df['Movimiento'] == 'LLEGADA']
    in_forecasted_rows = df[df['Movimiento'] == 'IN_FORECASTED']

    fila = df.iloc[0]
    info = {
        'Código': fila['Código'],
        'Nombre': fila['Nombre'],
        'NTécnico': fila['NTécnico']
    }

    if len(llegada_rows) == 0 or len(in_forecasted_rows) == 0:
        sin_datos.append(info)
        continue

    llegada = llegada_rows['Fecha'].iloc[0]
    in_forecasted = in_forecasted_rows['Fecha'].iloc[0]
    diferencia = llegada - in_forecasted

    if diferencia >= timedelta(seconds=30):
        retraso_llegada = llegada_rows['Retraso (segundos)'].iloc[0]
        retraso_in_forecasted = in_forecasted_rows['Retraso (segundos)'].iloc[0]
        retraso = retraso_llegada - retraso_in_forecasted

        info['Retraso'] = retraso
        posteriores.append(info)
    elif llegada < in_forecasted:
        no_posteriores.append(info)

df_posteriores = pd.DataFrame(posteriores)
df_no_posteriores = pd.DataFrame(no_posteriores)
df_sin_datos = pd.DataFrame(sin_datos)

In [ ]:
df_no_posteriores.sort_values(by='Código')

In [ ]:
df_no_posteriores

In [ ]:
df_posteriores

In [ ]:
sub_dfs = [grupo for _, grupo in df_posteriores.groupby(["Código"])]

In [ ]:
infor = []
for df in sub_dfs:
    media = df["Retraso"].mean()
    df["Retraso_promedio"] = media
    df["retraso_mayor_20"] = media > 20
    info = {
        'Código': df.iloc[0]['Código'],
        'Nombre': df.iloc[0]['Nombre'],
        'NTécnico': df.iloc[0]['NTécnico'],
        "Promedio_retraso > 20s": df["retraso_mayor_20"].iloc[0],
    }
    infor.append(info)
df_infor = pd.DataFrame(infor)
    
    

    

In [ ]:
data ={
    "prevision_posterior_llegada":df_no_posteriores,
    "Llegada_posterior_previson": df_infor,
}

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\2026_04_10_11_Llegada_prevision.xlsx")

In [ ]:
guardarExcelMulti(data,fname)

In [ ]:
C5_historico = historico_pro[(historico_pro["LíneaComercial"] == "C5") & (historico_pro["Núcleo"] == "MADRID")].copy()

In [ ]:
df_filtrado["Cod NUM_Tren"] = df_filtrado["Cod NUM_Tren"].astype(str)
df_filtrado[~df_filtrado["Cod NUM_Tren"].isin(historico_pro["NTécnico"])]

In [ ]:
C5_historico = C5_historico[C5_historico["NTécnico"].isin(df_filtrado["Cod NUM_Tren"])].copy()

In [ ]:
subdatagramas = [
    grupo for _, grupo in C5_historico.groupby(["NTécnico", "FechaOrigen"])
]

In [ ]:
limbo = []
valores_excluir = {"ELIMINACIÓN", "FIN"}

for subdatagrama in subdatagramas:
    if subdatagrama["Movimiento"].isin(valores_excluir).any():
        continue
    limbo.append(subdatagrama)


In [ ]:
ids = [subdatagrama["NTécnico"].iloc[0] for subdatagrama in limbo]

In [ ]:
ids

In [ ]:
limbo[0]

In [ ]:
historico_pro.sort_values(by=["FechaHora"] )

In [ ]:
fecha_inicio = pd.to_datetime('2026-01-13 00:00:00')
fecha_fin = pd.to_datetime('2026-02-04 00:00:00')

# Filtrar el DataFrame
df_filtrado = historico_pro[(historico_pro['FechaHora'] >= fecha_inicio) & (historico_pro['FechaHora'] <= fecha_fin)].copy()



In [ ]:
df_filtrado["TipoMovimiento"].unique()

In [ ]:
df_filtrado = df_filtrado.astype(str)

In [ ]:
Ru = ["XREG_GUADIANA_DEPARTURE","XREG_FORECAST_CHANGE_DESTINATION","XREG_FORECAST_CHANGE_ORIGIN","XREG_FORECAST_SECTION_INTERRUPTION","XREG_INTERMEDIATE_ORIGIN_FORECAST","XREG_SUPPRESSION"]

In [ ]:
df = historico_pro[historico_pro["TipoMovimiento"].isin(Ru)].copy()

In [ ]:
df["TipoMovimiento"].unique()

In [ ]:
rangos = [
    ("19500", "21999"),
    ("22000", "22999"),
    ("23000", "23999"),
    ("24000", "24999"),
    ("25000", "25999"),
    ("26000", "26999"),
    ("27000", "27999"),
    ("28000", "28999"),
    ("29000", "29999"),
    ("31000", "32999"),
    ("35000", "36499"),
    ("70000", "71099"),
    ("71100", "71599"),
    ("71600", "71799"),
    ("72000", "72099"),
    ("72200", "72499"),
    ("75000", "75999"),
    ("76000", "77999"),
]

# Convertir la columna Código a numérico (por si acaso)


# Crear condición para filtrar
condicion = False
for inicio, fin in rangos:
    condicion = condicion | ((df['NTécnico'] >= inicio) & (df['NTécnico'] <= fin))

# Filtrar el DataFrame
df_filtrado = df[condicion].copy()

In [ ]:
df_filtrado["FechaOrigen"] = pd.to_datetime(df_filtrado["FechaOrigen"],unit='ms',errors='coerce')

In [ ]:
df_filtrado["FechaOrigen"] = df_filtrado['FechaOrigen'].dt.strftime('%Y-%m-%d')


In [ ]:
df_filtrado["FechaOrigen"].unique()

In [ ]:
import yaml


# Ruta de la carpeta
carpeta = Path(r"data/Orden movimientos")

# Lista para guardar los datos
lista_yamls = []
for archivo in Path(carpeta).glob('*.y*ml'):
    with open(archivo, 'r', encoding='utf-8') as f:
        contenido = yaml.safe_load(f)
        
        # Combinar todos los valores de todas las claves del archivo
        todos_valores_archivo = []
        for valores in contenido.values():
            todos_valores_archivo.extend(valores)
        
        lista_yamls.append({
            'archivo': archivo.stem,
            'valores': todos_valores_archivo
        })

# Ver el resultado
for item in lista_yamls:
    print(f"Archivo: {item['archivo']}")
    print(f"Total valores: {len(item['valores'])}")
    print(f"Primeros valores: {item['valores'][:5]}")
    print("-" * 50)

In [ ]:
subdataframes = {}

for item in lista_yamls:
    nombre_archivo = item['archivo']
    codigos = item['valores']
    
    # Crear condiciones según TipoMovimiento
    condicion = (
        # Para XREG_FORECAST_CHANGE_ORIGIN y XREG_FORECAST_SECTION_INTERRUPTION → buscar en CódigoInicio
        ((df_filtrado['TipoMovimiento'].isin(['XREG_FORECAST_CHANGE_ORIGIN', 'XREG_FORECAST_SECTION_INTERRUPTION'])) & 
         (df_filtrado['CódigoIncio'].isin(codigos)))
        |
        # Para XREG_FORECAST_CHANGE_DESTINATION → buscar en CódigoFin
        ((df_filtrado['TipoMovimiento'] == 'XREG_FORECAST_CHANGE_DESTINATION') & 
         (df_filtrado['CódigoFin'].isin(codigos)))
        |
        # Para XREG_GUADIANA_DEPARTURE y XREG_INTERMEDIATE_ORIGIN_FORECAST → buscar en Código
        ((df_filtrado['TipoMovimiento'].isin(['XREG_GUADIANA_DEPARTURE', 'XREG_INTERMEDIATE_ORIGIN_FORECAST'])) & 
         (df_filtrado['Código'].isin(codigos)))
    )
    
    # Filtrar el DataFrame
    if ((nombre_archivo != "Lineas_AV") & (nombre_archivo != "trenes_ejemplo")):
        subdataframes[nombre_archivo] = df_filtrado[condicion]
# Ver resultados
for nombre, subdf in subdataframes.items():
    print(f"{nombre}: {len(subdf)} filas")

In [ ]:
Ru_1 = ["XREG_GUADIANA_DEPARTURE","XREG_FORECAST_CHANGE_DESTINATION","XREG_FORECAST_CHANGE_ORIGIN","XREG_FORECAST_SECTION_INTERRUPTION","XREG_INTERMEDIATE_ORIGIN_FORECAST"]

In [ ]:
resumen_data = []

for nombre, subdf in subdataframes.items():
    fila = {'Núcleo': nombre, 'Tota_Ru': len(subdf)}
    
    if len(subdf) > 0:
        conteo = subdf['TipoMovimiento'].value_counts()
        
        # Añadir cada tipo (incluso si es 0)
        for tipo in Ru_1:
            fila[tipo] = conteo.get(tipo, 0)
    else:
        # Si no hay filas, poner 0 en todos
        for tipo in Ru_1:
            fila[tipo] = 0
    
    resumen_data.append(fila)

df_resumen = pd.DataFrame(resumen_data)

In [ ]:
df_resumen

In [ ]:
Supresiones = df_filtrado[df_filtrado["TipoMovimiento"] == "XREG_SUPPRESSION"].copy()

In [ ]:
SupresionesOrigen = Supresiones[Supresiones["Secuencia"] == 1].copy()

In [ ]:
Supresiones = Supresiones[Supresiones["Secuencia"] != 1].copy()

In [ ]:
subdataframes_supresiones_origenes = {}


for item in lista_yamls:
    nombre_archivo = item['archivo']
    codigos = item['valores']
    if ((nombre_archivo != "Lineas_AV") & (nombre_archivo != "trenes_ejemplo")):
        subdataframes_supresiones_origenes[nombre_archivo] = SupresionesOrigen[SupresionesOrigen['Código'].isin(codigos)]

In [ ]:
subdataframes_Supresiones = {}


for item in lista_yamls:
    nombre_archivo = item['archivo']
    codigos = item['valores']
    if ((nombre_archivo != "Lineas_AV") & (nombre_archivo != "trenes_ejemplo")):
        subdataframes_Supresiones[nombre_archivo] = Supresiones[Supresiones['Código'].isin(codigos)]

In [ ]:
subdataframes_Supresiones.keys()

In [ ]:
conteo_data = []

for nombre, subdf in subdataframes_supresiones_origenes.items():
    conteo_data.append({
        'Núcleo': nombre,
        'Total_Supresiones_origen': len(subdf)
    })

df_supresiones_origen = pd.DataFrame(conteo_data)

In [ ]:
conteo_data = []

for nombre, subdf in subdataframes_Supresiones.items():
    conteo_data.append({
        'Núcleo': nombre,
        'Total_Supresiones': len(subdf)
    })

df_supresiones = pd.DataFrame(conteo_data)

In [ ]:
df_resumen

In [ ]:
df_resumen["Total_Ru"] = df_resumen["XREG_FORECAST_CHANGE_DESTINATION"] + df_resumen["XREG_FORECAST_CHANGE_ORIGIN"] + df_resumen["XREG_FORECAST_SECTION_INTERRUPTION"] + df_resumen["XREG_INTERMEDIATE_ORIGIN_FORECAST"]

In [ ]:
df_resumen.rename(columns = {"XREG_GUADIANA_DEPARTURE": "Total_Salida_Guadiana" }, inplace = True)

In [ ]:
df_resumen.drop(columns=["Tota_Ru"], inplace = True)


In [ ]:
df_resumen = df_resumen.merge(df_supresiones_origen, on="Núcleo", how="left").merge(df_supresiones, on="Núcleo", how="left")


In [ ]:
df_resumen = df_resumen[["Núcleo", "Total_Salida_Guadiana",  "Total_Supresiones_origen", "Total_Supresiones","Total_Ru","XREG_FORECAST_CHANGE_ORIGIN", "XREG_FORECAST_CHANGE_DESTINATION","XREG_FORECAST_SECTION_INTERRUPTION","XREG_INTERMEDIATE_ORIGIN_FORECAST"]].copy()

In [ ]:
df_resumen

In [ ]:
subdataframes_final = {}
subdataframes_final["Resumen"] = df_resumen

for item in lista_yamls:
    nombre_archivo = item['archivo']
    codigos = item['valores']
    
    # Crear condiciones según TipoMovimiento
    condicion = (
        # Para XREG_FORECAST_CHANGE_ORIGIN y XREG_FORECAST_SECTION_INTERRUPTION → buscar en CódigoInicio
        ((df_filtrado['TipoMovimiento'].isin(['XREG_FORECAST_CHANGE_ORIGIN', 'XREG_FORECAST_SECTION_INTERRUPTION'])) & 
         (df_filtrado['CódigoIncio'].isin(codigos)))
        |
        # Para XREG_FORECAST_CHANGE_DESTINATION → buscar en CódigoFin
        ((df_filtrado['TipoMovimiento'] == 'XREG_FORECAST_CHANGE_DESTINATION') & 
         (df_filtrado['CódigoFin'].isin(codigos)))
        |
        # Para XREG_GUADIANA_DEPARTURE y XREG_INTERMEDIATE_ORIGIN_FORECAST → buscar en Código
        ((df_filtrado['TipoMovimiento'].isin(['XREG_GUADIANA_DEPARTURE', 'XREG_INTERMEDIATE_ORIGIN_FORECAST','XREG_SUPPRESSION'])) & 
         (df_filtrado['Código'].isin(codigos)))
    )
    
    # Filtrar el DataFrame
    if ((nombre_archivo != "Lineas_AV") & (nombre_archivo != "trenes_ejemplo")):
        subdataframes_final[nombre_archivo] = df_filtrado[condicion]
# Ver resultados
for nombre, subdf in subdataframes_final.items():
    print(f"{nombre}: {len(subdf)} filas")

In [ ]:
for nombre, subdf in subdataframes_final.items():
    print(f"{nombre}: {len(subdf)} filas")

In [ ]:
Fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Ru.xlsx")
guardarExcelMulti(subdataframes_final, Fname)

In [ ]:
subdatagramas = [
    grupo for _, grupo in df.groupby(["NTécnico", "FechaOrigen", "Código"])
]


In [ ]:
test = []

for sub_df in subdatagramas:
    movimientos = sub_df["Movimiento"]
    if ("FIN" in movimientos.values) and ("ELIMINACIÓN" in movimientos.values):
        test.append(sub_df)


In [ ]:
test[2]

In [ ]:
df.columns

In [ ]:
df["Núcleo"].unique()

In [ ]:
test = df[df["Núcleo"] == "MADRID"].copy()

In [ ]:
test["LíneaComercial"].unique()

In [ ]:
C1_history = test[test["LíneaComercial"] == "C1M"].reset_index(drop=True).copy()

In [ ]:
C1_history

In [ ]:
códigos = C1_history["Código"].unique().tolist()


In [ ]:
# from src.api.APIs import hacerPeticion


# def getCirculacionesComerciales(codigo: str) -> pd.DataFrame:
#     """
#     Carga las siguientes salidas de la estación seleccionada
#     """
#     HOSTPATH = "http://info.api.elcano.operaciones.adif/portroyalmanager/circulationpaths/departures/traffictype/"

#     movimientos = []
#     page = 0
#     while True:
#         print(f"\r{codigo} - Página {page}", end="", flush=True)
#         data = {
#             "commercialService": "BOTH",
#             "commercialStopType": "BOTH",
#             "page": {"pageNumber": page},
#             "stationCode": codigo,
#             "trafficType": "ALL",
#         }
#         data = json.dumps(data)

#         response = hacerPeticion(
#             "POST",
#             HOSTPATH,
#             data=data,
#         )
#         if not response:
#             break
#         response = response.json()
#         movimientos.extend(response["commercialPaths"])
#         print(movimientos)
#         page += 1
#         # if page == response["totalPages"]:
#         #     break
#     rename_cols = {
#         # "commercialPathInfo.timestamp": "Fecha",
#         "commercialPathInfo.commercialPathKey.commercialCirculationKey.commercialNumber": "NComercial",
#         "commercialPathInfo.commercialPathKey.commercialCirculationKey.launchingDate": "FechaOrigen",
#         "commercialPathInfo.commercialPathKey.originStationCode": "CódigoOrigen",
#         "commercialPathInfo.commercialPathKey.destinationStationCode": "CódigoDestino",
#         # "commercialPathInfo.commercialPathKey.line": "Línea",
#         "commercialPathInfo.core": "Núcleo",
#         "commercialPathInfo.line": "Línea",
#         "commercialPathInfo.observation": "Observaciones",
#         "commercialPathInfo.trafficType": "Tráfico",
#         "commercialPathInfo.opeProComPro.operator": "Operador",
#         "commercialPathInfo.opeProComPro.product": "Producto",
#         "commercialPathInfo.opeProComPro.commercialProduct": "ProductoComercial",
#         "passthroughStep.stopType": "TipoParada",
#         "passthroughStep.stationCode": "Código",
#         "passthroughStep.arrivalPassthroughStepSides.plannedTime": "LlegadaPlanificada",
#         "passthroughStep.arrivalPassthroughStepSides.forecastedOrAuditedDelay": "RetrasoLlegada",
#         "passthroughStep.arrivalPassthroughStepSides.timeType": "TipoLlegada",
#         "passthroughStep.arrivalPassthroughStepSides.plannedPlatform": "VíaLlegadaPlanificada",
#         "passthroughStep.arrivalPassthroughStepSides.sitraPlatform": "VíaLlegadaSitra",
#         "passthroughStep.arrivalPassthroughStepSides.ctcPlatform": "VíaLlegadaCTC",
#         # "passthroughStep.arrivalPassthroughStepSides.resultantPlatform":"",
#         "passthroughStep.arrivalPassthroughStepSides.circulationState": "EstadoLlegada",
#         "passthroughStep.arrivalPassthroughStepSides.technicalCirculationKey.technicalNumber": "NTécnicoLlegada",
#         # "passthroughStep.arrivalPassthroughStepSides.technicalCirculationKey.technicalLaunchingDate":"FechaOrigenLlegada",
#         "passthroughStep.departurePassthroughStepSides.plannedTime": "SalidaPlanificada",
#         "passthroughStep.departurePassthroughStepSides.forecastedOrAuditedDelay": "RetrasoSalida",
#         "passthroughStep.departurePassthroughStepSides.timeType": "TipoSalida",
#         "passthroughStep.departurePassthroughStepSides.plannedPlatform": "VíaSalidaPlanificada",
#         "passthroughStep.departurePassthroughStepSides.sitraPlatform": "VíaSalidaSitra",
#         "passthroughStep.departurePassthroughStepSides.ctcPlatform": "VíaSalidaCTC",
#         # "passthroughStep.departurePassthroughStepSides.resultantPlatform":"",
#         "passthroughStep.departurePassthroughStepSides.circulationState": "EstadoSalida",
#         "passthroughStep.departurePassthroughStepSides.technicalCirculationKey.technicalNumber": "NTécnicoSalida",
#         # "passthroughStep.departurePassthroughStepSides.technicalCirculationKey.technicalLaunchingDate":"FechaOrigenSalida",
#     }
#     df_movimientos = pd.json_normalize(movimientos)
#     df_movimientos = df_movimientos[
#         [el for el in rename_cols.keys() if el in df_movimientos.columns]
#     ].rename(columns=rename_cols)

#     # Carga de los nombres de estación
#     info_estaciones = pd.read_excel("data/info_estaciones.xlsx")[["Código", "Nombre"]]
#     info_estaciones["Código"] = info_estaciones["Código"].apply(lambda x: f"{x:0>5}")
#     map_codigo_nombre = dict(info_estaciones.values)

#     # Formatear info
#     date_cols = ["FechaOrigen", "LlegadaPlanificada", "SalidaPlanificada"]
#     df_movimientos[[c for c in date_cols if c in df_movimientos.columns]] = pd.concat(
#         parallelizeFunction(
#             lambda x: x.map(time2localtime, unit="ms"),
#             data=splitDataframe(
#                 df_movimientos[[c for c in date_cols if c in df_movimientos.columns]],
#                 1000,
#             ),
#             show_progress=True,
#             desc="Formateando fechas.",
#             output="series",
#         )
#     )
#     df_movimientos[["NombreOrigen", "NombreDestino", "Nombre"]] = pd.concat(
#         parallelizeFunction(
#             lambda x: x.map(map_codigo_nombre.get),
#             data=splitDataframe(
#                 df_movimientos[["CódigoOrigen", "CódigoDestino", "Código"]], 1000
#             ),
#             show_progress=True,
#             desc="Formateando fechas.",
#             output="series",
#         )
#     )
#     return df_movimientos



In [ ]:
import pandas as pd
import json
import re
from src.api.APIs import hacerPeticion

day = "2026-01-29"
HOSTPATH = f"http://info.api.elcano.operaciones.adif/planning/{day}/findAllCirculationsPlanningDay"

# GET request usually uses query parameters, not JSON body
params = {"day": pd.to_datetime(day).strftime("%Y-%m-%d")}

response = hacerPeticion("GET", HOSTPATH, data=params)

# Check if response is valid
if response is None:
    raise ValueError("hacerPeticion returned None!")

# Clean and parse response
res_data = response.json()

In [ ]:
import pandas as pd
from zoneinfo import ZoneInfo  # para manejo de zona horaria

rows = []

for item in res_data:
    circulation_id = item["circulationId"]["number"]
    launch_date = item["circulationId"]["launchingDate"]
    launch_date_str = f"{launch_date[0]:04d}-{launch_date[1]:02d}-{launch_date[2]:02d}"
    
    day_train = item["dayTrain"]
    commercial_number = day_train["commercialNumber"]
    company = day_train["company"]
    operator = day_train["operator"]
    train_type = day_train["trainType"]
    line = day_train["line"]["name"] if day_train["line"] else None
    nucleus = day_train["nucleus"]
    
    journey = day_train.get("journey", {})
    steps = journey.get("steps", [])
    
    for step in steps:
        # Convertir timestamps a datetime en UTC primero
        arrive_dt = pd.to_datetime(step.get("arrive"), unit="ms", utc=True)
        leave_dt = pd.to_datetime(step.get("leave"), unit="ms", utc=True)
        
        # Convertir a hora de Madrid
        madrid_tz = ZoneInfo("Europe/Madrid")
        arrive_dt = arrive_dt.astimezone(madrid_tz)
        leave_dt = leave_dt.astimezone(madrid_tz)
        
        row = {
            "circulationId": circulation_id,
            "launch_date": launch_date_str,
            "company": company,
            "operator": operator,
            "line": line,
            "nucleus": nucleus,
            "trainType": train_type,
            "step": step.get("step"),
            "arrive": arrive_dt,
            "leave": leave_dt,
            "pointId": step.get("pointId"),
            "distanceToPrevious": step.get("distanceToPrevious"),
            "parkingTrack": step.get("parkingTrack"),
            "parkingTrackForDeparture": step.get("parkingTrackForDeparture"),
            "stationaryType": step.get("stationaryType"),
            "parity": step.get("parity"),
        }
        rows.append(row)

# Crear DataFrame
df = pd.DataFrame(rows)



In [ ]:
renamed_columns = {
    "circulationId": "NTécnico",
    "launch_date": "FechaOrigen", 
    "company": "Compañia",
    "operator": "Operador",
    "line":"Línea",
    "nucleus":"Núcleo",
    "trainType": "TipoTren",
    "step": "Secuencia",
    "arrive": "Llegada",
    "leave": "Salida",
    "pointId": "Código",
    "distanceToPrevious": "DistanciaRespectoAnterior",
    "parkingTrack": "VíaEstacionamiento",
    "parkingTrackForDeparture": "VíaSalida",
    "stationaryType": "TipoEstacionamiento",
    "parity": "Paridad",
    "technicalStop": "ParadaTécnica",
    "circulationMode": "ModoCirculación"
}

df.rename(columns=renamed_columns, inplace=True)


In [ ]:
C1_planificadas = df[df["NTécnico"].isin(C1_history["NTécnico"])].copy()

In [ ]:
C1_planificadas

In [ ]:
lista_subdfs = [subdf for _, subdf in C1_planificadas.groupby(["NTécnico"])]

In [ ]:
import pandas as pd

def calcular_tiempo_recorrido_llegada_salida(df):
    """
    Calcula el tiempo de recorrido usando columnas Llegada y Salida
    """
    df = df.copy()
    
    # Asegurarnos de que las fechas están en formato datetime
    df['Llegada'] = pd.to_datetime(df['Llegada'])
    df['Salida'] = pd.to_datetime(df['Salida'])
    
    # Ordenar por secuencia
    df = df.sort_values('Secuencia').reset_index(drop=True)
    
    # Inicializar columna de tiempo de recorrido en minutos
    df['tiempo_recorrido_min'] = 0.0
    
    # Para cada fila desde la segunda en adelante
    for i in range(1, len(df)):
        # Obtener la Salida de la fila anterior
        salida_anterior = df.loc[i-1, 'Salida']
        # Obtener la Llegada de la fila actual
        llegada_actual = df.loc[i, 'Llegada']
        
        # Calcular diferencia en minutos
        tiempo_minutos = (llegada_actual - salida_anterior).total_seconds() / 60
        df.loc[i, 'tiempo_recorrido_min'] = tiempo_minutos
    
    # Convertir a formato mm:ss
    def minutos_a_mmss(minutos):
        if pd.isna(minutos) or minutos == 0:
            return "00:00"
        minutos_enteros = int(minutos)
        segundos = int((minutos - minutos_enteros) * 60)
        return f"{minutos_enteros:02d}:{segundos:02d}"
    
    df['RecorridoPlanificado'] = df['tiempo_recorrido_min'].apply(minutos_a_mmss)
    
    # Opcional: eliminar columna auxiliar
    # df = df.drop('tiempo_recorrido_min', axis=1)
    
    return df

# Aplicar al dataframe
lista_planificadas = [calcular_tiempo_recorrido_llegada_salida(df) for df in lista_subdfs]

In [ ]:
planificada = pd.concat(lista_planificadas,ignore_index=True)

In [ ]:
planificada

In [ ]:
C1_planificadas_filtrada = planificada[["FechaOrigen","NTécnico","Código","Llegada","Salida","RecorridoPlanificado"]].copy()

In [ ]:
C1_planificadas_filtrada

In [ ]:
C1_history["Fecha_sin_hora"] = C1_history["Fecha"].dt.date

In [ ]:
C1_history["FuenteVía"].unique()

In [ ]:
lista_subdfs = [subdf for _, subdf in C1_history.groupby(["NTécnico", "Fecha_sin_hora"])]


In [ ]:
movimientos_filtrados = []

for subdf in lista_subdfs:
    temp_df = subdf[
        (subdf["FuenteVía"] == "CTC") &
        (subdf["Movimiento"].isin(["ORIGEN", "LLEGADA", "SALIDA", "FIN"]))
    ]
    
    # Quedarse solo con la última fila de cada movimiento
    temp_df = temp_df.drop_duplicates(subset=["Código","Movimiento"], keep="last")
    
    movimientos_filtrados.append(temp_df)

In [ ]:
import pandas as pd

movimientos_finales = []

for subdf in movimientos_filtrados:
    subdf["Secuencia"] = pd.to_numeric(subdf["Secuencia"])
    
    # Ordenar por Secuencia y Movimiento según el orden lógico
    orden_mov = ["ORIGEN", "LLEGADA", "SALIDA", "FIN"]
    subdf["Movimiento"] = pd.Categorical(subdf["Movimiento"], categories=orden_mov, ordered=True)
    subdf = subdf.sort_values(["Secuencia", "Movimiento"]).reset_index(drop=True)
    
    # Agrupar por Secuencia
    grupos = subdf.groupby("Secuencia")
    
    resumen_secuencia = grupos.agg(
        Fecha_inicio=("Fecha", "first"),  # ORIGEN/LLEGADA de la secuencia
        Fecha_fin=("Fecha", "last")       # SALIDA/FIN de la secuencia
    ).reset_index()
    
    # Calcular retraso con la secuencia siguiente
    resumen_secuencia["Fecha_inicio_siguiente"] = resumen_secuencia["Fecha_inicio"].shift(-1)
    resumen_secuencia["Retraso"] = resumen_secuencia["Fecha_inicio_siguiente"] - resumen_secuencia["Fecha_fin"]
    
    # Convertir a mm:ss
    resumen_secuencia["Retraso_mmss"] = resumen_secuencia["Retraso"].apply(
        lambda x: f"{int(x.total_seconds()//60):02d}:{int(x.total_seconds()%60):02d}" if pd.notnull(x) else None
    )
    
    movimientos_finales.append(resumen_secuencia)


In [ ]:
movimientos_finales[0]


In [ ]:

movimientos_filtrados[0].head(20)


In [ ]:
def calcular_tiempo_recorrido(df):
    """
    Calcula el tiempo de recorrido para un dataframe individual
    """
    df = df.copy()
    df['Fecha'] = pd.to_datetime(df['Fecha'])
    df = df.sort_values('Secuencia').reset_index(drop=True)
    
    # Crear columna con fecha de SALIDA/ORIGEN
    df['fecha_salida_ref'] = df.apply(
        lambda row: row['Fecha'] if row['Movimiento'] in ['SALIDA', 'ORIGEN'] else None, 
        axis=1
    ).ffill()
    
    # Calcular tiempo de recorrido en minutos
    df['tiempo_recorrido_minutos'] = 0.0
    if len(df) > 1:
        df.loc[1:, 'tiempo_recorrido_minutos'] = (
            (df.loc[1:, 'Fecha'] - df.loc[1:, 'fecha_salida_ref']).dt.total_seconds() / 60
        )
    df.loc[0, 'tiempo_recorrido_minutos'] = 0
    
    # Convertir a formato mm:ss
    def minutos_a_mmss(minutos):
        if pd.isna(minutos) or minutos == 0:
            return "00:00"
        minutos_enteros = int(minutos)
        segundos = int((minutos - minutos_enteros) * 60)
        return f"{minutos_enteros:02d}:{segundos:02d}"
    
    df['tiempo_recorrido'] = df['tiempo_recorrido_minutos'].apply(minutos_a_mmss)
    
    # Limpiar columnas auxiliares
    df = df.drop(['fecha_salida_ref', 'tiempo_recorrido_minutos'], axis=1)
    
    return df

# Procesar lista de subdatagramas
lista_resultados = [calcular_tiempo_recorrido(df) for df in movimientos_filtrados]

# Ver resultados
for i, df_resultado in enumerate(lista_resultados):
    print(f"\n=== Subdatagrama {i+1} ===")
    print(df_resultado[['Secuencia', 'NombreOrigen', 'Movimiento', 'Fecha', 'tiempo_recorrido']])

In [ ]:
lista_resultados_filter = []
for df in lista_resultados:
    df_1  = df[["Fecha","NTécnico","Nombre","Código","Secuencia","Movimiento","tiempo_recorrido","FechaOrigen","Fecha_sin_hora"]].copy()
    df_2 = df_1[
        ((df_1['Secuencia'] == 1) & (df_1['Movimiento'] == 'ORIGEN')) |
        (df_1['Movimiento'] == 'LLEGADA')
    ].reset_index(drop=True)
    df_2.drop(columns="Movimiento",inplace=True)
#     df_pivot = df_2.pivot_table(
#     index='Secuencia',
#     columns='Fecha_sin_hora',
#     values='tiempo_recorrido',
#     aggfunc='first'  # En caso de duplicados, toma el primero
# )
    
    lista_resultados_filter.append(df_2)
    

In [ ]:
lista_resultados_filter[0]

In [ ]:
recorrido = pd.concat(lista_resultados_filter,ignore_index=True)

In [ ]:
recorrido_completo = pd.merge(
    C1_planificadas_filtrada[["NTécnico","Código","RecorridoPlanificado"]],
    recorrido,
    left_on=["NTécnico","Código"],
    right_on=["NTécnico","Código"],
    how = "right"
)

In [ ]:
recorrido_completo["Fecha_sin_hora"] = recorrido_completo["Fecha_sin_hora"].astype(str)

In [ ]:
# Con secuencia y nombre como índice
df_pivot = recorrido_completo.pivot_table(
    index=['NTécnico', 'Nombre',"Código","Secuencia","RecorridoPlanificado"],
    columns='Fecha_sin_hora',
    values='tiempo_recorrido',
    aggfunc='first'
).reset_index()


In [ ]:
df_pivot.columns.name = None

In [ ]:
df_pivot

In [ ]:
df_pivot.columns = df_pivot.columns.astype(str)

In [ ]:
def verificar_dataframe_para_excel(df):
    """
    Verifica que el DataFrame esté listo para guardar en Excel
    """
    print("="*60)
    print("VERIFICACIÓN DE DATAFRAME PARA EXCEL")
    print("="*60)
    
    # 1. Verificar nombres de columnas
    print("\n1. NOMBRES DE COLUMNAS:")
    no_strings = [col for col in df.columns if not isinstance(col, str)]
    if no_strings:
        print(f"   ❌ Columnas que NO son string: {no_strings}")
        print(f"      Tipos: {[type(col).__name__ for col in no_strings]}")
    else:
        print("   ✅ Todas las columnas son strings")
    
    # 2. Verificar columnas duplicadas
    print("\n2. COLUMNAS DUPLICADAS:")
    duplicados = df.columns[df.columns.duplicated()].tolist()
    if duplicados:
        print(f"   ❌ Columnas duplicadas: {duplicados}")
    else:
        print("   ✅ No hay columnas duplicadas")
    
    # 3. Verificar tipos de datos problemáticos
    print("\n3. TIPOS DE DATOS:")
    print(df.dtypes)
    
    # 4. Verificar valores nulos
    print("\n4. VALORES NULOS:")
    nulos = df.isnull().sum()
    if nulos.sum() > 0:
        print(f"   ⚠️ Columnas con valores nulos:")
        print(nulos[nulos > 0])
    else:
        print("   ✅ No hay valores nulos")
    
    # 5. Verificar tamaño del DataFrame
    print("\n5. DIMENSIONES:")
    print(f"   Filas: {len(df)}")
    print(f"   Columnas: {len(df.columns)}")
    if len(df) > 1048576:
        print("   ❌ Excede el límite de filas de Excel (1,048,576)")
    if len(df.columns) > 16384:
        print("   ❌ Excede el límite de columnas de Excel (16,384)")
    
    # 6. Verificar caracteres especiales en nombres de columnas
    print("\n6. CARACTERES ESPECIALES EN COLUMNAS:")
    especiales = [col for col in df.columns if any(c in str(col) for c in ['/', '\\', '?', '*', '[', ']', ':'])]
    if especiales:
        print(f"   ⚠️ Columnas con caracteres especiales: {especiales}")
    else:
        print("   ✅ No hay caracteres especiales problemáticos")
    
    print("\n" + "="*60)
    
    return len(no_strings) == 0 and len(duplicados) == 0

# Usar la función
df_ok = verificar_dataframe_para_excel(df)
if df_ok:
    print("🎉 DataFrame listo para guardar en Excel")
else:
    print("⚠️ Corrige los problemas antes de guardar")
def limpiar_dataframe_para_excel(df):
    """
    Limpia y prepara el DataFrame para Excel automáticamente
    """
    df = df.copy()
    
    # 1. Convertir nombres de columnas a string
    df.columns = df.columns.astype(str)
    
    # 2. Limpiar nombres de columnas (quitar caracteres problemáticos)
    df.columns = df.columns.str.replace('[/\\?*\[\]:]', '_', regex=True)
    
    # 3. Eliminar columnas.name si existe
    df.columns.name = None
    
    # 4. Convertir tipos problemáticos
    for col in df.columns:
        # Si la columna es de tipo object pero contiene timestamps, convertir a string
        if df[col].dtype == 'object':
            if any(isinstance(x, pd.Timestamp) for x in df[col].dropna()):
                df[col] = df[col].astype(str)
    
    # 5. Reemplazar infinitos
    df = df.replace([float('inf'), float('-inf')], pd.NA)
    
    return df

# Usar la función
df_limpio = limpiar_dataframe_para_excel(df_pivot)
verificar_dataframe_para_excel(df_limpio)

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\TiempoRecorrido_C1.xlsx")

In [ ]:
data={
    "C1":df_pivot
}

In [ ]:
guardarExcelMulti(data, fname)

In [ ]:
suprimidos =[]
for sub_dfs  in lista_subdfs:
   if 'Movimiento' in sub_dfs.columns and 'ELIMINACIÓN' in sub_dfs['Movimiento'].values:
        suprimidos.append(sub_dfs)      

In [ ]:
Maniobra = []
tipos_maniobra = {"MANIOBRA_SALIDA", "MANIOBRA_APROXIMACION", "MANIOBRA_LLEGADA"}

for df in suprimidos:
    df = df.sort_values(["NTécnico", "Secuencia"]).reset_index(drop=True)
    df_maniobra = pd.DataFrame()

    for tren, grupo in df.groupby("NTécnico"):
        # Filas de eliminación
        eliminaciones = grupo[grupo["Movimiento"] == "ELIMINACIÓN"]

        for _, elim_row in eliminaciones.iterrows():
            seq_elim = elim_row["Secuencia"]

            # Filas posteriores al movimiento de eliminación dentro del mismo grupo (NTécnico)
            posteriores = grupo[grupo["Secuencia"] > seq_elim]

            if not posteriores.empty:
                # Filtramos solo las filas de maniobra
                posteriores_maniobra = posteriores[posteriores["Movimiento"].isin(tipos_maniobra)]

                # Si existe al menos un movimiento de maniobra después de la eliminación en la misma estación
                if not posteriores_maniobra.empty:
                    df_maniobra = pd.concat(
                        [df_maniobra, elim_row.to_frame().T, posteriores_maniobra],
                        ignore_index=True
                    )

    # Verificamos que el dataframe no esté vacío antes de agregarlo
    if not df_maniobra.empty:
        Maniobra.append(df_maniobra)


In [ ]:
Maniobra

In [ ]:
Maniobra[2]

<H1> CTC CAF</H1>

In [ ]:
Fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\configuracion_CVs_CAF_mensaje6.xlsx")

In [ ]:
diccionario_hojas = pd.read_excel(Fname, sheet_name=None)


In [ ]:
lista_dataframes = list(diccionario_hojas.values())[1:]

In [ ]:
for df in lista_dataframes:
    df["MNemonico"] = df["identificador"].str[:2]
    df["Elemento"] = df["identificador"].str[2:]

In [ ]:
lista_dataframes[1]

In [ ]:
ruta_base = Path(r"C:\topogen-adif-repo\baseline")

In [ ]:
import re
import os
patron = re.compile(
    r"recta\('.*?','(.*?)',.*?,'[^']*'\)",
    re.IGNORECASE
)

In [ ]:
from collections import defaultdict
indice_mnemonicos = defaultdict(list)   # { mnem : [archivo1, archivo2] }

for root, dirs, files in os.walk(ruta_base):
    for f in files:
        if f.endswith(".pl"):
            path_archivo = os.path.join(root, f)

            with open(path_archivo, "r", encoding="cp1252", errors="ignore") as file:
                contenido = file.read()

            # Extraer TODOS los 'MN' encontrados en recta(...)
            coincidencias = patron.findall(contenido)

            # Añadir al índice
            for mn in coincidencias:
                indice_mnemonicos[mn].append(path_archivo)


In [ ]:
indice_mnemonicos

In [ ]:
for df in lista_dataframes:
    df["estacion"] = None

    for i, fila in df.iterrows():
        mnem = fila["MNemonico"]

        if mnem in indice_mnemonicos:
            # Tomamos el primer archivo encontrado
            ruta_archivo = indice_mnemonicos[mnem][0]

            # Guardar SOLO el nombre del archivo (no la ruta completa)
            df.at[i, "estacion"] = os.path.basename(ruta_archivo)

In [ ]:
for df in lista_dataframes:
    # Quitar extensión .pl
    df["estacion"] = df["estacion"].str.replace(r"\.pl$", "", regex=True)

In [ ]:
lista_dataframes[1]

In [ ]:
nombres_hojas = ["Leon", "Oviedo", "Bilbao", "Santander", "Orense", "Orense_Viejo", "Valencia"]

diccionario_hojas = {nombre: df for nombre, df in zip(nombres_hojas, lista_dataframes)}
    

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Traducción_estación.xlsx")

In [ ]:
guardarExcelMulti(diccionario_hojas,fname)

In [ ]:
estaciones = loadEstaciones()

In [ ]:
estaciones["Tecnólogo"].unique()

In [ ]:
import json
import pandas as pd
from pathlib import Path

fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Lista_tren.json")

# Abrir como texto primero y detectar codificación
with open(fname, 'rb') as f:
    raw = f.read()

# Intentar decodificar automáticamente
for enc in ['utf-8', 'utf-16', 'utf-16-le', 'utf-16-be', 'latin1']:
    try:
        text = raw.decode(enc)
        print(f"Archivo decodificado correctamente con {enc}")
        break
    except UnicodeDecodeError:
        continue

# Intentar cargar JSON línea por línea
data_list = []
for line in text.splitlines():
    line = line.strip()
    if line:  # Ignorar líneas vacías
        try:
            data_list.append(json.loads(line))
        except json.JSONDecodeError:
            # Algunas líneas podrían no ser JSON; ignorarlas o procesarlas manualmente
            print("Línea no válida:", line[:50])

# Convertir a DataFrame
df = pd.DataFrame(data_list)
print(df.head())


In [ ]:
import json
import pandas as pd
from pathlib import Path

fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Lista_tren.json")

# Leer como binario para detectar encoding
with open(fname, 'rb') as f:
    raw = f.read()

# Intentar decodificación
for enc in ['utf-8', 'utf-16', 'utf-16-le', 'utf-16-be', 'latin1']:
    try:
        text = raw.decode(enc)
        print(f"Archivo decodificado correctamente con {enc}")
        break
    except UnicodeDecodeError:
        continue


# Procesar línea por línea
records = []
for line in text.splitlines():
    line = line.strip()
    if not line:
        continue
    try:
        obj = json.loads(line)
        records.append(obj)
    except json.JSONDecodeError:
        print("Línea no válida:", line[:80])


# Normalizar JSON anidado → columnas planas
df = pd.json_normalize(records, max_level=3)

print(df.head())
print(df.columns)


In [ ]:
import json
from pathlib import Path

fname1 = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Listado.txt")
fname2 = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Listado_fixed.json")

def fix_json_file(input_path, output_path):
    with open(input_path, "r", encoding="utf-16") as f:
        raw = f.read()

    objects = []
    buffer = ""
    depth = 0
    inside = False

    for ch in raw:
        if ch == "{":
            depth += 1
            inside = True

        if inside:
            buffer += ch

        if ch == "}":
            depth -= 1
            if depth == 0 and inside:
                # fin de un objeto JSON completo
                try:
                    obj = json.loads(buffer)
                    objects.append(obj)
                except Exception as e:
                    print("Error decodificando JSON:", e)
                    print("Contenido problemático:\n", buffer[:200], "...")
                buffer = ""
                inside = False

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(objects, f, indent=2, ensure_ascii=False)

    print(f"✔ Archivo generado: {output_path}")
    print(f"✔ Objetos JSON encontrados: {len(objects)}")


fix_json_file(fname1, fname2)


In [ ]:
fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Listado_fixed.json")

In [ ]:
df = pd.read_json(fname)

df_flat = pd.json_normalize(df.to_dict(orient="records"))

In [ ]:
df_flat.columns = (
    df_flat.columns
      .str.replace(r"^header\.", "", regex=True)
      .str.replace(r"^messageType\.", "", regex=True)
)
df_flat["timestampMSG"] = pd.to_datetime(df_flat["timestampMSG"], unit="ms")
df_flat["timeStampCTC"] = pd.to_datetime(df_flat["timeStampCTC"], unit="ms")

In [ ]:
fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Listado_tren_sin_técnico_11.xlsx")
guardarExcel(df_flat, fname)

In [ ]:
df_flat

In [ ]:
df

<h1> mensaje MIE 6 </h1>


In [ ]:
start_date = "2025-11-24"
end_date = "2025-11-26"
estaciones = []

In [ ]:
estaciones = []
ntrenes = [rellenarId(el) for el in np.arange(100000)]
# ntrenes = [rellenarId(el) for el in np.arange(2000, 6000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=True,
    jCTC=False,
    pro=False,
)
historico_pro = historico_pro.sort_values(
    by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
).reset_index(drop=True)

# Añadir información de la fecha
historico_pro["Día"] = historico_pro["Fecha"].dt.date
historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
historico_pro

In [ ]:
Proyección = historico_pro[historico_pro["Elemento"] == "PROYECCIÓN"].copy()

In [ ]:
estaciones = loadEstaciones()

In [ ]:
estaciones[estaciones["Código"] == "08208"]

In [ ]:
list = ["EST1","NOE","NOO1","NOO2","OUR1","NOR1","NOR2","OURP"]
CTC = estaciones[estaciones["Catálogo"].isin(list)]

In [ ]:
CTC[CTC["Código"] == "08208"]

In [ ]:
estaciones_caf = CTC["Código"].drop_duplicates().tolist()

In [ ]:
estaciones_caf.append("05242")

In [ ]:
if "08208"  in estaciones_caf:
    print("Sí, el código 32004 está en la lista de estaciones CAF.")

In [ ]:
test = getEstacionamientos(["32004"])


In [ ]:
test

In [ ]:
estacionamientos = getEstacionamientos(estaciones_caf)

In [ ]:
estacionamientos["Código"] = estacionamientos["Código"].replace("05242", "05243")

In [ ]:
estacionamientos[~estacionamientos["Código"].isin(estaciones_caf)]

In [ ]:
estacionamientos[estacionamientos["Código"] == "32004"]

In [ ]:
test = pd.merge(
    Proyección,
    estacionamientos[["Código","Vía","VíaTécnica"]],
    left_on=["Código","Vía"],
    right_on=["Código","Vía"],
    how = "left"
)

In [ ]:
no_esta = test[test["VíaTécnica"].isna()]

In [ ]:
no_esta["Código"].unique()

In [ ]:
test2 = pd.merge(
    Proyección,
    estacionamientos[["Código","Vía","VíaTécnica"]],
    left_on=["Código","Vía"],
    right_on=["Código","Vía"],
    how = "right"
)

In [ ]:
no_proyección = test2[test2["NTécnico"].isna()]

In [ ]:
no_proyección.columns

In [ ]:
no_proyección_1 = no_proyección[["Código","VíaTécnica","Vía"]]

In [ ]:
CTC

In [ ]:
regex = r'\s+(RAM|AV|RC)$'
CTC['Nombre'] = CTC['Nombre'].str.replace(regex, '', regex=True)


In [ ]:
CTC.drop_duplicates(subset="Código",inplace=True)

In [ ]:
no_proyección_final = pd.merge(
    no_proyección_1,
    CTC[["Código","Nombre","CTC","NombreCTC"]]
)

In [ ]:
no_proyección_final = no_proyección_final[["CTC","NombreCTC","Código","Nombre","Vía","VíaTécnica"]]

In [ ]:
fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\graylog\mie\All-Messages-search-result.csv")


In [ ]:
mie_procesor = MIEProcessor()
mie = mie_procesor.readLogFile(fname)

In [ ]:
df_mie = mie_procesor.loadLogFile(fname)

In [ ]:
df_mie

In [ ]:
df_mie = pd.merge(
    df_mie,
    estaciones[["CTC","NombreCTC","Código","Nombre","Mnemónico"]],
    on  = "Mnemónico",
    how="left"  
)

In [ ]:
df_mie["Fecha"] = pd.to_datetime(df_mie["Fecha"], unit= "ms")

In [ ]:
df_mie['Fecha'] = pd.to_datetime(df_mie['Fecha']).dt.floor('S')

In [ ]:
df_mie

In [ ]:
df_mie_1 = df_mie[(df_mie["Fecha"] >= pd.to_datetime("2025-11-24")) & (df_mie["Fecha"] < pd.to_datetime("2025-11-26"))]

In [ ]:
df_mie_1

In [ ]:
df_tmp = df_mie_1.merge(
    estacionamientos[['Código', 'Vía']],
    on='Código',
    how='left'
)

df_filtrado = df_tmp[df_tmp['Elemento'] == df_tmp['Vía']]

In [ ]:
df_filtrado

In [ ]:
estacionamientos

In [ ]:
df_mie_final = df_filtrado.merge(
    estacionamientos,
    on = ["Código","Vía"],
    how = "right"
   
)

In [ ]:
Sin_mie = df_mie_final[df_mie_final["NTécnico"].isna()]

In [ ]:
Sin_mie = Sin_mie[["Código","Vía","VíaTécnica"]].copy()

In [ ]:
estaciones

In [ ]:
df_mie_final_1 = Sin_mie.merge(
    estaciones[["CTC","Código","Nombre","NombreCTC"]],
    on = "Código",
    how = "left"
)


In [ ]:
df_mie_final_1 = df_mie_final_1[["CTC","NombreCTC","Código","Nombre","Vía","VíaTécnica"]]

In [ ]:
data = {
    "Vía_estacionamiento_no_MSE": no_proyección_final,
    "Vía_estaconamiento_no_MIE": df_mie_final_1
}

In [ ]:
fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\24-25_nomviembre_Vía_estacionamiento_sin_proyección.xlsx")

In [ ]:
guardarExcelMulti(data, fname)

<h1> Estaciones Origenes </h1>

In [ ]:
estaciones = loadEstaciones()
estaciones_sin_ctc = loadEstacionSinCTC()

In [ ]:
estaciones_conctc = estaciones[["CTC","NombreCTC","Código","Nombre"]].copy()

In [ ]:
estaciones_sin_ctc["SinCTC"] = True

In [ ]:
estaciones_sin_ctc = estaciones_sin_ctc[["Código","Delegación","SinCTC","Nombre"]].copy()

In [ ]:
estaciones_sin_ctc

In [ ]:
circulaciones_planificadas = getCirculacionesPlanificadas("2025-12-05")

In [ ]:
circulaciones_planificadas

In [ ]:
circulaciones_planificadas[circulaciones_planificadas["NTécnico"]=="17191"]

In [ ]:
no_comercial = ["T.L.E.","Material Vacio","Mercancias","Servicio Interno","Transporte excepcional","Maquina Aislada Mercancias","MAQUINA AISLADA","Maquina Aislada","Material vacio RAM","Mercancias RAM"]

In [ ]:
circulaciones_planificadas= circulaciones_planificadas[~circulaciones_planificadas["TipoTren"].isin(no_comercial)]

In [ ]:
subdatagramas = [grupo for _, grupo in circulaciones_planificadas.groupby("NTécnico")]

In [ ]:
subdatagramas[0]


In [ ]:
origenes = []
for df in subdatagramas:
    origen = df[df["Secuencia"] == 1]
    origenes.append(origen)
    

In [ ]:
origeness  =pd.concat(origenes)

In [ ]:
origenes_1= origeness.drop_duplicates(subset="Código")

In [ ]:
estaciones = pd.concat([estaciones_conctc,estaciones_sin_ctc],ignore_index=True)

In [ ]:
estaciones

In [ ]:
origenes = estaciones[estaciones["Código"].isin(origeness["Código"])]

In [ ]:
origenes["SinCTC"].fillna(False, inplace=True)

In [ ]:
origenes = origenes[["CTC","NombreCTC","Código","Nombre","SinCTC"]]

In [ ]:
origenes[origenes["Código"] == "50200"]

In [ ]:
circulaciones_planificadas[(circulaciones_planificadas["Código"] == "50200") & (circulaciones_planificadas["Secuencia"] == 1)]

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\origenes_3.xlsx")
guardarExcel(origenes,fname)

<h1> Supresiones </h1>

In [ ]:
start_date = "2025-11-27"
end_date = "2025-11-28"
estaciones = []
xREG=True
ntrenes = [rellenarId(el) for el in np.arange(100000)]
# ntrenes = [rellenarId(el) for el in np.arange(2000, 6000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=False,
    jCTC=False,
    pro=True,
    xREG=xREG,
)
if (xREG == False):
    historico_pro = historico_pro.sort_values(
        by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
    ).reset_index(drop=True)

    # Añadir información de la fecha
    historico_pro["Día"] = historico_pro["Fecha"].dt.date
    historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
    historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
    historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
xreg = historico_pro.copy()

In [ ]:
start_date = "2025-11-27"
end_date = "2025-11-28"
estaciones = []
xREG=False
ntrenes = [rellenarId(el) for el in np.arange(100000)]
# ntrenes = [rellenarId(el) for el in np.arange(2000, 6000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=True,
    jCTC=False,
    pro=True,
    xREG=xREG,
)
if (xREG == False):
    historico_pro = historico_pro.sort_values(
        by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
    ).reset_index(drop=True)

    # Añadir información de la fecha
    historico_pro["Día"] = historico_pro["Fecha"].dt.date
    historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
    historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
    historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
xreg = xreg[
        (xreg["FechaHora"] >= pd.to_datetime(start_date))
        & (xreg["FechaHora"] <= pd.to_datetime(end_date))
    ].copy()

In [ ]:
supresiones = historico_pro[historico_pro["Movimiento"] == "ELIMINACIÓN"]

<h3>Supresiones en Origen</h3>

In [ ]:
sub_dfs = [grupo for _, grupo in supresiones.groupby("NTécnico")]

In [ ]:
supresiones_origen = [
    df
    for df in sub_dfs
    if (df["Secuencia"] == 1).any()
]


In [ ]:
def timedelta_to_hhmmss(td):
    if pd.isna(td):
        return None  # o "" o "0:00:00", como prefieras

    total_seconds = int(td.total_seconds())
    sign = "-" if total_seconds < 0 else ""
    total_seconds = abs(total_seconds)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60

    return f"{sign}{hours:02d}:{minutes:02d}:{seconds:02d}"

In [ ]:
origen = []
for df in supresiones_origen:
    df = df[["FechaOrigen","NTécnico","Producto","Código","Nombre","SalidaPlanificada","Fecha"]].copy()
    df.rename(columns={"Fecha":"FechaSupresión"}, inplace=True)
    df["Antelación"] = df["SalidaPlanificada"] - df["FechaSupresión"]
    df["Antelación"] = df["Antelación"].apply(timedelta_to_hhmmss)
    origen.append(df)
    

In [ ]:
origen[0]

In [ ]:
origenes = pd.concat(origen,ignore_index=True)

In [ ]:
origenes = origenes[["Código","Nombre","NTécnico","Producto","SalidaPlanificada","FechaSupresión","Antelación"]]

<h3>Supresiones Cambio destino</h3>

In [ ]:
no_origen = supresiones[~supresiones["NTécnico"].isin(origenes["NTécnico"])].copy()

In [ ]:
no_origen.columns

In [ ]:
destino = xreg[xreg["TipoMovimiento"] == "XREG_FORECAST_CHANGE_DESTINATION"].copy()

In [ ]:
interrupcion = xreg[xreg["TipoMovimiento"] == "XREG_FORECAST_SECTION_INTERRUPTION"].copy()

In [ ]:
no_origen.columns

In [ ]:
no_origen_clear = no_origen[["FechaOrigen","NTécnico","Código","Nombre","SalidaPlanificada","Secuencia","Producto"]].copy()

In [ ]:
start_date = pd.to_datetime(start_date)
mask = no_origen_clear["SalidaPlanificada"].dt.date == (start_date - pd.Timedelta(days=1)).date()

In [ ]:
no_origen_clear.loc[mask, "SalidaPlanificada"] = no_origen_clear.loc[mask, "SalidaPlanificada"].apply(
    lambda x: start_date + pd.Timedelta(hours=x.hour, minutes=x.minute, seconds=x.second)
)

In [ ]:
suprimido_destinos = pd.merge(
    no_origen_clear,
    destino[["FechaHora","NTécnico","Nombre","Código","SecuenciaFin"]],
    left_on=["NTécnico","Secuencia"],
    right_on =["NTécnico","SecuenciaFin"],
    how = "left"
)

In [ ]:
suprimido_destinos_1 = suprimido_destinos.dropna(subset=["SecuenciaFin"]).copy()

In [ ]:
suprimido_destinos_1.drop(columns=["Código_y","Nombre_y","SecuenciaFin"],inplace = True)

In [ ]:
suprimido_destinos_1.rename (columns={"Código_x":"Código","Nombre_x":"Nombre","FechaHora":"HoraAviso"},inplace=True)

In [ ]:
suprimido_destinos_1["Incidencia"] = "Cambio destino"

In [ ]:

suprimido_destinos_1 = suprimido_destinos_1[["Código","Nombre","Producto","Incidencia","SalidaPlanificada","HoraAviso"]]

In [ ]:
suprimido_destinos_1["Antelación"]  = suprimido_destinos_1["SalidaPlanificada"]-suprimido_destinos_1["HoraAviso"]

In [ ]:
suprimido_destinos_1["Antelación"] = suprimido_destinos_1["Antelación"].apply(timedelta_to_hhmmss)

In [ ]:
suprimido_destinos_1.reset_index(drop=True,inplace=True)

In [ ]:
suprimido_interrupcion = pd.merge(
    no_origen_clear,
    interrupcion[["NTécnico","FechaHora","CódigoIncio"]],
    left_on=["NTécnico","Código"],
    right_on=["NTécnico","CódigoIncio"],
    how = "left"
)

In [ ]:
suprimido_interrupcion_1 = suprimido_interrupcion[~suprimido_interrupcion["FechaHora"].isna()].copy()


In [ ]:
suprimido_interrupcion_1["Antelación"] = suprimido_interrupcion_1["SalidaPlanificada"]- suprimido_interrupcion_1["FechaHora"]

In [ ]:
suprimido_interrupcion_1["Antelación"] = suprimido_interrupcion_1["Antelación"].apply(timedelta_to_hhmmss)

In [ ]:
suprimido_interrupcion_1["Incidencia"] = "Interrupción"

In [ ]:
suprimido_interrupcion_1 = suprimido_interrupcion_1[["Código", "Nombre","Producto","Incidencia","SalidaPlanificada","FechaHora","Antelación"]].rename(columns={"FechaHora":"HoraAviso"}).copy()

In [ ]:
suprimido_interrupcion

In [ ]:
supresiones = xreg[xreg["TipoMovimiento"] == "XREG_SUPPRESSION"].copy()

In [ ]:
sin_aviso = supresiones[(~supresiones["NTécnico"].isin(destino["NTécnico"])) & (~supresiones["NTécnico"].isin(interrupcion["NTécnico"])) & (supresiones["Secuencia"] != 1)]

In [ ]:
sin_aviso_1 = no_origen[(~no_origen["NTécnico"].isin(destino["NTécnico"])) & (~no_origen["NTécnico"].isin(interrupcion["NTécnico"]))]

In [ ]:
sin_aviso_2 = pd.merge(
    sin_aviso[["FechaHora","NTécnico","Código"]],
    sin_aviso_1[["NTécnico","Código","Nombre","Producto","SalidaPlanificada"]],
    on = ["NTécnico","Código"],
    how = "left"
)

In [ ]:
sin_aviso_total = sin_aviso_2[~sin_aviso_2["SalidaPlanificada"].isna()].copy()

In [ ]:
sin_aviso_total.reset_index(drop=True,inplace=True)

In [ ]:
sin_aviso_total["Incidencia"] = "Cambio destino"

In [ ]:
sin_aviso_total["HoraAviso"] = "No anunciada"

In [ ]:
sin_aviso_total["Antelación"] = sin_aviso_total["SalidaPlanificada"] - sin_aviso_total["FechaHora"]

In [ ]:
sin_aviso_total["Antelación"] =sin_aviso_total["Antelación"].apply(timedelta_to_hhmmss)

In [ ]:
sin_aviso_total = sin_aviso_total[["Código","Nombre","Producto","Incidencia","SalidaPlanificada","HoraAviso","FechaHora","Antelación"]].rename(columns={"FechaHora":"HoraSupresión"})

In [ ]:
tabla2 = pd.concat(
    [suprimido_destinos_1, suprimido_interrupcion_1, sin_aviso_total],
    ignore_index=True
)

In [ ]:
tabla2 = tabla2[["Código","Nombre","Producto","Incidencia","SalidaPlanificada","HoraAviso","HoraSupresión","Antelación"]]

In [ ]:
tabla2

<h3> Cambio Origen </h3>

In [ ]:
cambio_origen = xreg[xreg["TipoMovimiento"] == "XREG_FORECAST_CHANGE_ORIGIN"].copy()

In [ ]:
guadiana = xreg[xreg["TipoMovimiento"] == "XREG_GUADIANA_DEPARTURE"]

In [ ]:
cambio_origen_1 = cambio_origen[cambio_origen["NTécnico"].isin(guadiana["NTécnico"])]


In [ ]:
cambio_origen2 = pd.merge(
    cambio_origen_1[["FechaHora","NTécnico","CódigoIncio"]],
    historico_pro[["NTécnico","Código","Nombre","Producto","SalidaPlanificada"]],
    left_on=["NTécnico","CódigoIncio"],
    right_on=["NTécnico","Código"],
    how = "left"
)

In [ ]:
cambio_origen2["Incidencia"] ="Cambio origen"

In [ ]:
cambio_origen2 = cambio_origen2[["Código","Nombre","Producto","Incidencia","SalidaPlanificada","FechaHora"]].rename(columns={"FechaHora":"HoraAviso"})

In [ ]:
cambio_origen2["Antelación"] = None

In [ ]:
# cambio_origen2["Antelación"] = cambio_origen2["Antelación"].apply(timedelta_to_hhmmss)

In [ ]:
cambio_origen_no_aviso = guadiana[(~guadiana["NTécnico"].isin(cambio_origen["NTécnico"])) & (~guadiana["NTécnico"].isin(interrupcion["NTécnico"]))]

In [ ]:
cambio_origen3 = pd.merge(
    cambio_origen_no_aviso[["FechaHora","NTécnico","Código"]],
    historico_pro[["NTécnico","Código","Nombre","Producto","SalidaPlanificada"]],
    left_on=["NTécnico","Código"],
    right_on=["NTécnico","Código"],
    how = "left"
)

In [ ]:
cambio_origen3["Incidencia"] = "Cambio origen"

In [ ]:
cambio_origen3 = cambio_origen3[["Código","Nombre","Producto","Incidencia","SalidaPlanificada","FechaHora"]].rename(columns={"FechaHora":"HoraSalidaGuadiana"})
cambio_origen3["Antelación"] = None

In [ ]:
cambio_origen3

In [ ]:
interupcion_con_aviso = interrupcion[interrupcion["NTécnico"].isin(guadiana["NTécnico"])]

In [ ]:
interupcion_con_aviso_1 = pd.merge(
    interupcion_con_aviso[["FechaHora","NTécnico","CódigoFin"]],
    historico_pro[["NTécnico","Código","Nombre","Producto","SalidaPlanificada"]],
    left_on= ["NTécnico","CódigoFin"],
    right_on= ["NTécnico","Código"],
    how = "left"
)

In [ ]:
interupcion_con_aviso_1["Incidencia"] = "Interrupción"

In [ ]:
interupcion_con_aviso_1 = interupcion_con_aviso_1[["Código","Nombre","Producto","Incidencia","SalidaPlanificada","FechaHora"]].rename(columns={"FechaHora":"HoraAviso"})
interupcion_con_aviso_1["Antelación"] = None

In [ ]:
interupcion_con_aviso_1

In [ ]:
tabla3 = pd.concat([cambio_origen2,interupcion_con_aviso_1,cambio_origen3],ignore_index=True)

In [ ]:
tabla3 = tabla3[["Código","Nombre","Producto","Incidencia","SalidaPlanificada","HoraAviso","HoraSalidaGuadiana","Antelación"]]

<h3> Supresiones Destino</h3>


In [ ]:
lista_subdfs = [group for _, group in historico_pro.groupby(['NTécnico', 'FechaOrigen'])]
destino = []

# Recorremos cada sub-DataFrame
for sub_dfs in lista_subdfs:
    sub_dfs_sorted = sub_dfs.sort_values(by='Secuencia', ascending=True)
    ultima_fila = sub_dfs_sorted.iloc[-1]
    if ultima_fila['Código'] == ultima_fila['CódigoDestino']:
        destino.append(ultima_fila)

# Mostramos la lista de últimas secuencias
df_destino = pd.DataFrame(destino)
df = df_destino[(df_destino["Movimiento"] == "ELIMINACIÓN") & (df_destino["Secuencia"] > 1)]

In [ ]:
data = {
    "Supresiones_origen":origenes,
    "Supresiones_cambio_destino":tabla2,
    "CambioOrigen":tabla3,
    "SupresionesDestino":df
}
fname= Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\27-11-2025_Supresiones.xlsx")

In [ ]:
guardarExcelMulti(data,fname)

In [ ]:
tabla2

In [ ]:
tabla2[tabla2["Incidencia"] =="Interrupción"]

In [ ]:
Fname= Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Paso_Chamartin.json")
with open(Fname, 'r') as f:
    data = json.load(f)
    # Extraemos la lista de caminos comerciales
    commercial_paths = data["commercialPaths"]

# Lista para almacenar los datos desanidados
    flattened_data = []

# Iteramos sobre cada "commercialPath" y desanidamos la información
    for path in commercial_paths:
        commercial_path_info = path["commercialPathInfo"]
        passthrough_step = path["passthroughStep"]
    
        # Extraemos datos comerciales
        commercial_data = {
            "timestamp": commercial_path_info["timestamp"],
            "commercialNumber": commercial_path_info["commercialPathKey"]["commercialCirculationKey"]["commercialNumber"],
            "launchingDate": commercial_path_info["commercialPathKey"]["commercialCirculationKey"]["launchingDate"],
            "originStationCode": commercial_path_info["commercialOriginStationCode"],
            "destinationStationCode": commercial_path_info["commercialDestinationStationCode"],
            "line": commercial_path_info["line"],
            "core": commercial_path_info["core"],
            "trafficType": commercial_path_info["trafficType"],
            "operator": commercial_path_info["opeProComPro"]["operator"],
            "product": commercial_path_info["opeProComPro"]["product"],
            "commercialProduct": commercial_path_info["opeProComPro"]["commercialProduct"],
            "compositionLenghtType": commercial_path_info["compositionData"]["compositionLenghtType"],
            "compositionFloorType":commercial_path_info["compositionData"]["compositionFloorType"],
            "compositionAccesible": commercial_path_info["compositionData"]["accesible"]
        }
    
        # Extraemos datos del "passthroughStep"
        passthrough_data = {
            "stopType": passthrough_step["stopType"],
            "announceable": passthrough_step["announceable"],
            "stationCode": passthrough_step["stationCode"],
            "plannedTime": passthrough_step["arrivalPassthroughStepSides"]["plannedTime"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "forecastedOrAuditedDelay": passthrough_step["arrivalPassthroughStepSides"]["forecastedOrAuditedDelay"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "timeType": passthrough_step["arrivalPassthroughStepSides"]["timeType"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "plannedPlatform": passthrough_step["arrivalPassthroughStepSides"]["plannedPlatform"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "sitraPlatform": passthrough_step["arrivalPassthroughStepSides"]["sitraPlatform"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "ctcPlatform": passthrough_step["arrivalPassthroughStepSides"]["ctcPlatform"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "viewPlatform": passthrough_step["arrivalPassthroughStepSides"]["viewPlatform"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "operatorPlatform": passthrough_step["arrivalPassthroughStepSides"]["operatorPlatform"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "platform": passthrough_step["arrivalPassthroughStepSides"]["platform"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "resultantPlatform": passthrough_step["arrivalPassthroughStepSides"]["resultantPlatform"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "preassignedPlatform": passthrough_step["arrivalPassthroughStepSides"]["preassignedPlatform"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "guadiana": passthrough_step["arrivalPassthroughStepSides"]["guadiana"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "originChange": passthrough_step["arrivalPassthroughStepSides"]["originChange"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "destinationChange": passthrough_step["arrivalPassthroughStepSides"]["destinationChange"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "observation": passthrough_step["arrivalPassthroughStepSides"]["observation"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "circulationState": passthrough_step["arrivalPassthroughStepSides"]["circulationState"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "announceState": passthrough_step["arrivalPassthroughStepSides"]["announceState"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "technicalNumber":passthrough_step["arrivalPassthroughStepSides"]["technicalCirculationKey"]["technicalNumber"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "technicalLaunchingDate":passthrough_step["arrivalPassthroughStepSides"]["technicalCirculationKey"]["technicalLaunchingDate"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "technicalSequence": passthrough_step["arrivalPassthroughStepSides"]["technicalSequence"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "inmediateDeparture": passthrough_step["arrivalPassthroughStepSides"]["visualEffects"]["inmediateDeparture"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "countDown": passthrough_step["arrivalPassthroughStepSides"]["visualEffects"]["countDown"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "showDelay": passthrough_step["arrivalPassthroughStepSides"]["visualEffects"]["showDelay"] if passthrough_step.get("arrivalPassthroughStepSides") else None,
            "departurePassthroughStepSides": passthrough_step["departurePassthroughStepSides"]

        }
    
        # Combinamos los datos comerciales y del paso
        flattened_data.append({**commercial_data, **passthrough_data})

    # Convertimos la lista de diccionarios en un DataFrame de pandas
    df = pd.DataFrame(flattened_data)



In [ ]:
df.columns

In [ ]:
rename ={
    "timestamp":"Fecha",
    "commercialNumber":"NTécnico Comercial",
    "launchingDate":"FechaOrigen",
    "originStationCode":"CódigoOrigen",
    "destinationStationCode":"CódigoDestino",
    "line":"Línea",
    "core":"Núcleo",
    "trafficType":"TipoTráfico",
    "operator":"Operador",
    "product":"Producto",
    "commercialProduct":"ProductoComercial",
    "compositionLenghtType":"TipoLongitudComposición",
    "compositionFloorType":"TipoSueloTren",
    "compositionAccesible":"ComposiciónAccesible",
    "stopType":"TipoStop",
    "announceable":"anunciable",
    "stationCode":"CódigoEstación",
    "plannedTime":"FechaPlanificada",
    "forecastedOrAuditedDelay":"RetrasoPrevisto/Auditado",
    "timeType":"TipoTiempo",
    "plannedPlatform":"VíaPlanificada",
    "sitraPlatform":"VíaSitra",
    "ctcPlatform":"VíaCTC",
    "viewPlatform":"VíaView",
    "operatorPlatform":"VíaOperador",
    "platform":"Vía",
    "resultantPlatform":"VíaResultante",
    "preassignedPlatform":"Vía Preasignado",
    "guadiana":"Guadiana",
    "originChange":"CambioOrigen",
    "destinationChange":"CambioDestino",
    "observation":"Observación",
    "circulationState":"EstadoCirculación",
    "announceState":"EstadoAnuncio",
    "technicalNumber":"NTécnico",
    "technicalLaunchingDate":"FechaCirculaciónTécnica",
    "technicalSequence":"SecuenciaTécnica",
    "inmediateDeparture":"SalidaInmediata",
    "countDown":"CuentaAtrás",
    "showDelay":"MostrarRetraso",
    "departurePassthroughStepSides":"LadoPasoSalida"
}

In [ ]:
df.rename(columns = rename, inplace=True)

In [ ]:
def convertir_timestamp_columna(df, columna_timestamp):
    """
    Convierte una columna de timestamps Unix en milisegundos a fecha legible en un DataFrame.
    
    Parámetros:
    - df: DataFrame de pandas.
    - columna_timestamp: Nombre de la columna que contiene los timestamps en milisegundos.
    
    Devuelve:
    - DataFrame con la columna convertida a fecha legible.
    """
    # Convertir la columna de timestamps en milisegundos a segundos y luego a fecha legible
    df[columna_timestamp] = pd.to_datetime(df[columna_timestamp] / 1000, unit='s')
    df[columna_timestamp] = df[columna_timestamp].dt.floor('s')
    return df

In [ ]:
df = convertir_timestamp_columna(df,"Fecha")

In [ ]:
df = convertir_timestamp_columna(df,"FechaOrigen")

In [ ]:
df = convertir_timestamp_columna(df,"FechaPlanificada")
df = convertir_timestamp_columna(df,"FechaCirculaciónTécnica")

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Vista_Paso_Chamartin.xlsx")

In [ ]:
guardarExcel(df,fname)

<H1> Dependencias RC - RAM - AV </H1>

In [ ]:
estaciones_ctc = loadEstaciones()

In [ ]:
estaciones_ctc["ConCTC"]= True

In [ ]:
estaciones_sin_ctc = loadEstacionSinCTC()

In [ ]:
estaciones_sin_ctc["ConCTC"] = False

In [ ]:
estaciones = pd.concat([estaciones_ctc,estaciones_sin_ctc],ignore_index=True)

In [ ]:
estaciones

In [ ]:
duplicado = estaciones[estaciones["Código"].duplicated(keep= False)].copy()

In [ ]:
duplicado.sort_values(by="Código", inplace=True)

In [ ]:
duplicado.reset_index(drop=True, inplace=True)

In [ ]:
duplicado

In [ ]:
subdirección = pd.read_csv("data/Subdirección.csv")

In [ ]:
subdirección

In [ ]:
duplicado1 = pd.merge(
    duplicado,
    subdirección,
    on = "Código",
    how="left"
)

In [ ]:
duplicado1

In [ ]:
duplicado1 = duplicado1[["Subdirección","ConCTC","CTC","NombreCTC","Código","Nombre"]]

In [ ]:
duplicado1

In [ ]:
fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Dependencias_rc_ram_av_1.xlsx")

In [ ]:
guardarExcel(duplicado1,fname)

In [ ]:
ambito =  pd.read_csv("data/Ámbitos.csv",sep = ";")

In [ ]:
ambito

In [ ]:
guardarExcel(ambito,Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Ámbitos.xlsx"))

<h1> Comparativa MSE Y SITRA </h1>

In [ ]:
estaciones  = loadEstaciones()

In [ ]:
BCN = estaciones[estaciones["CTC"] == "BCN"].copy()

In [ ]:
estaciones_bcn = BCN["Código"].tolist()

In [ ]:
estaciones_bcn

In [ ]:
start_date = "2025-12-17"
end_date = "2025-12-18"
estaciones =[]
ntrenes = [rellenarId(el) for el in np.arange(100000)]
# ntrenes = [rellenarId(el) for el in np.arange(2000, 6000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=True,
    jCTC=False,
    pro=True,
)
historico_pro = historico_pro.sort_values(
    by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
).reset_index(drop=True)

# Añadir información de la fecha
historico_pro["Día"] = historico_pro["Fecha"].dt.date
historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
xsiv = historico_pro.copy()

In [ ]:
start_date = "2025-12-17"
end_date = "2025-12-18"
estaciones =[]
ntrenes = [rellenarId(el) for el in np.arange(100000)]
# ntrenes = [rellenarId(el) for el in np.arange(2000, 6000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=False,
    jCTC=False,
    xREG= True,
    pro=True,
)
# historico_pro = historico_pro.sort_values(
#     by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
# ).reset_index(drop=True)

# # Añadir información de la fecha
# historico_pro["Día"] = historico_pro["Fecha"].dt.date
# historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
# historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
# historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
xreg = historico_pro.copy()

In [ ]:
xreg["FechaOrigen"].unique()

In [ ]:
xsiv_bcn = xsiv[xsiv["Código"].isin(estaciones_bcn)].copy()

In [ ]:
xsiv_bcn

In [ ]:
xreg_bcn = xreg[xreg["Código"].isin(estaciones_bcn)].copy()

In [ ]:
xreg["TipoMovimiento"].unique()

In [ ]:
xsiv_bcn_llegada_salida = xsiv_bcn[xsiv_bcn["Movimiento"].isin(["LLEGADA","SALIDA"])].copy()

In [ ]:
no_comercial = ["T.L.E.","Material Vacio","Mercancias","Servicio Interno","Transporte excepcional","Maquina Aislada Mercancias","MAQUINA AISLADA","Maquina Aislada","Material vacio RAM","Mercancias RAM"]

In [ ]:
xsiv_bcn_llegada_salida = xsiv_bcn_llegada_salida[~xsiv_bcn_llegada_salida["Producto"].isin(no_comercial)].copy()

In [ ]:
xsiv_bcn_llegada_salida.drop_duplicates(subset=["NTécnico","FechaOrigen","Código","Movimiento"], keep='first', inplace=True)

In [ ]:
xreg_bcn_llegada_salida = xreg_bcn[xreg_bcn["TipoMovimiento"].isin(["XREG_ARRIVAL","XREG_DEPARTURE"])].copy()

In [ ]:
xreg_bcn_llegada_salida["FechaOrigen"] = pd.to_datetime(xreg_bcn_llegada_salida["FechaOrigen"],unit="ms")
xreg_bcn_llegada_salida['FechaOrigen'] = pd.to_datetime(xreg_bcn_llegada_salida['FechaOrigen'], unit='ms', utc=True)

# Convertir a hora de Madrid
xreg_bcn_llegada_salida['FechaOrigen'] = xreg_bcn_llegada_salida['FechaOrigen'].dt.tz_convert('Europe/Madrid')

In [ ]:
xreg_bcn_llegada_salida["FechaOrigen"] = pd.to_datetime(xreg_bcn_llegada_salida["FechaOrigen"]).dt.strftime('%Y-%m-%d')

In [ ]:
xreg_bcn_llegada_salida.drop_duplicates(subset=["NTécnico","FechaOrigen","Código","TipoMovimiento"], keep='first', inplace=True)

In [ ]:
xreg_bcn_llegada_salida_filtrado = xreg_bcn_llegada_salida[["FechaHora","NTécnico","Código","Nombre","TipoMovimiento","Secuencia","FechaOrigen"]].copy()

In [ ]:
xreg_bcn_llegada_salida_filtrado.loc[xreg_bcn_llegada_salida_filtrado["TipoMovimiento"] == "XREG_ARRIVAL", "TipoMovimiento"] = "LLEGADA"
xreg_bcn_llegada_salida_filtrado.loc[xreg_bcn_llegada_salida_filtrado["TipoMovimiento"] == "XREG_DEPARTURE", "TipoMovimiento"] = "SALIDA"

In [ ]:
xreg_bcn_llegada_salida_filtrado

In [ ]:
xsiv_bcn_llegada_salida['FechaOrigen'] = pd.to_datetime(xsiv_bcn_llegada_salida['FechaOrigen'])
xreg_bcn_llegada_salida_filtrado['FechaOrigen'] = pd.to_datetime(xreg_bcn_llegada_salida_filtrado['FechaOrigen'])
mapeado = pd.merge(
    xsiv_bcn_llegada_salida,
    xreg_bcn_llegada_salida_filtrado,
    left_on=["FechaOrigen","NTécnico","Código","Movimiento"],
    right_on=["FechaOrigen","NTécnico","Código","TipoMovimiento"],
    how = "left",
    suffixes=("_SIV","_REG")
)

In [ ]:
mapeado["Retraso en (Segundo)"] = (mapeado["Fecha"] - mapeado["FechaHora"]).dt.total_seconds()

In [ ]:
mapeado


In [ ]:
mapeado_filtrado = mapeado[["Fecha","NTécnico","Código","Nombre_SIV","Movimiento","Secuencia_SIV","FechaHora","Secuencia_REG","Retraso en (Segundo)","FechaOrigen"]].copy()

In [ ]:
mapeado_filtrado.rename(columns={"Fecha":"Fecha_MSE","Nombre_SIV":"Nombre","Secuencia_SIV":"Secuencia_MSE","FechaHora":"Fecha_SITRA","Secuencia_REG":"Secuencia_SITRA"},inplace=True)

In [ ]:
mapeado_filtrado[mapeado_filtrado["NTécnico"] == "25470"]

In [ ]:
xreg_bcn_llegada_salida_filtrado[xreg_bcn_llegada_salida_filtrado["NTécnico"] == "25470"]

In [ ]:
mapeado_filtrado

In [ ]:
xreg_bcn_llegada_salida_filtrado[xreg_bcn_llegada_salida_filtrado["NTécnico"] == "90406"]

In [ ]:
mapeado_filtrado[mapeado_filtrado["Código"] == "79409"]

In [ ]:
# import pandas as pd
# import json
# import os

# # --- Preparar datos ---
# mapeado_filtrado['Retraso en (Segundo)'] = pd.to_numeric(
#     mapeado_filtrado['Retraso en (Segundo)'], errors='coerce'
# )
# df_con_retraso = mapeado_filtrado[mapeado_filtrado['Retraso en (Segundo)'].notna()]

# stats_por_estacion = df_con_retraso.groupby('Nombre').agg({
#     'Retraso en (Segundo)': 'mean',
#     'Código': 'first'
# }).reset_index()
# stats_por_estacion.columns = ['Estacion', 'Retraso_Promedio', 'Codigo']
# stats_por_estacion = stats_por_estacion.sort_values('Retraso_Promedio', ascending=True)

# # --- Convertir DataFrame a JSON ---
# df_valid = mapeado_filtrado.dropna(subset=['Fecha_SITRA'])
# df_json = df_valid.to_json(orient='records', date_format='iso')

# # --- Ruta de guardado ---
# base_path = r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\dashboard_retraso_estacion"
# os.makedirs(base_path, exist_ok=True)

# # --- Página Principal (índice) ---
# html_index = f"""
# <!DOCTYPE html>
# <html lang="es">
# <head>
#     <meta charset="UTF-8">
#     <meta name="viewport" content="width=device-width, initial-scale=1.0">
#     <title>Dashboard - Retraso por Estación</title>
#     <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
#     <style>
#         body {{
#             font-family: Arial, sans-serif;
#             margin: 20px;
#             background-color: #f5f5f5;
#         }}
#         h1 {{
#             color: #333;
#             text-align: center;
#         }}
#         .container {{
#             background: white;
#             padding: 20px;
#             border-radius: 8px;
#             box-shadow: 0 2px 4px rgba(0,0,0,0.1);
#         }}
#         .info {{
#             background: #e3f2fd;
#             padding: 10px;
#             border-radius: 4px;
#             margin-bottom: 20px;
#             text-align: center;
#         }}
#     </style>
# </head>
# <body>
#     <div class="container">
#         <h1>📊 Retraso Promedio por Estación</h1>
#         <div class="info">
#             💡 <strong>Haz clic en cualquier barra</strong> para ver el análisis detallado de esa estación
#         </div>
#         <div id="fig_barras"></div>
#     </div>

#     <script>
#         var stats = {json.dumps(stats_por_estacion.to_dict(orient='records'))};

#         var barra = {{
#             y: stats.map(r => r.Codigo + " - " + r.Estacion),
#             x: stats.map(r => r.Retraso_Promedio),
#             type: 'bar',
#             orientation: 'h',
#             marker: {{
#                 color: stats.map(r => r.Retraso_Promedio > 0 ? '#4CAF50' : '#f44336')
#             }},
#             text: stats.map(r => r.Retraso_Promedio.toFixed(2) + "s"),
#             textposition: 'outside',
#             hovertemplate: '<b>%{{y}}</b><br>Retraso Promedio: %{{x:.2f}} seg<br><i>Click para ver detalles</i><extra></extra>'
#         }};

#         var layout_barras = {{
#             title: '',
#             xaxis: {{title: 'Retraso Promedio (segundos)'}},
#             yaxis: {{title: 'Estación', automargin: true}},
#             height: Math.max(400, stats.length * 30),
#             margin: {{l: 200}}
#         }};

#         Plotly.newPlot('fig_barras', [barra], layout_barras);

#         // Evento click para redirigir
#         document.getElementById('fig_barras').on('plotly_click', function(data) {{
#             var codigo = data.points[0].y.split(' - ')[0];
#             var filename = 'Dependencias_' + codigo + '.html';
#             window.location.href = filename;
#         }});
#     </script>
# </body>
# </html>
# """

# # Guardar página principal
# with open(os.path.join(base_path, "index.html"), "w", encoding="utf-8") as f:
#     f.write(html_index)

# # --- Crear páginas individuales para cada estación ---
# for idx, row in stats_por_estacion.iterrows():
#     codigo = row['Codigo']
#     estacion = row['Estacion']
#     retraso = row['Retraso_Promedio']
    
#     html_estacion = f"""
# <!DOCTYPE html>
# <html lang="es">
# <head>
#     <meta charset="UTF-8">
#     <meta name="viewport" content="width=device-width, initial-scale=1.0">
#     <title>Análisis - {codigo} {estacion}</title>
#     <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
#     <style>
#         body {{
#             font-family: Arial, sans-serif;
#             margin: 20px;
#             background-color: #f5f5f5;
#         }}
#         .container {{
#             background: white;
#             padding: 20px;
#             border-radius: 8px;
#             box-shadow: 0 2px 4px rgba(0,0,0,0.1);
#         }}
#         .header {{
#             display: flex;
#             justify-content: space-between;
#             align-items: center;
#             margin-bottom: 20px;
#         }}
#         .btn-back {{
#             background: #2196F3;
#             color: white;
#             padding: 10px 20px;
#             border: none;
#             border-radius: 4px;
#             cursor: pointer;
#             text-decoration: none;
#             display: inline-block;
#         }}
#         .btn-back:hover {{
#             background: #1976D2;
#         }}
#         .stats {{
#             background: #f0f0f0;
#             padding: 15px;
#             border-radius: 4px;
#             margin-bottom: 20px;
#         }}
#         h1 {{
#             color: #333;
#             margin: 0;
#         }}
#     </style>
# </head>
# <body>
#     <div class="container">
#         <div class="header">
#             <h1>🚉 {codigo} - {estacion}</h1>
#             <a href="index.html" class="btn-back">← Volver al Dashboard</a>
#         </div>
        
#         <div class="stats">
#             <strong>Retraso Promedio:</strong> {retraso:.2f} segundos
#         </div>
        
#         <h2>Comparación Temporal MSE vs SITRA</h2>
#         <div id="fig_comparacion"></div>
#     </div>

#     <script>
#         var df = {df_json};
#         var codigo = '{codigo}';
        
#         // Filtrar datos de esta estación
#         var df_filtrado = df.filter(d => d.Código === codigo);
        
#         if (df_filtrado.length === 0) {{
#             document.getElementById('fig_comparacion').innerHTML = 
#                 '<p style="text-align:center;color:#999;">No hay datos disponibles para esta estación</p>';
#         }} else {{
#             var data = [];
            
#             ['LLEGADA', 'SALIDA'].forEach(function(mov, i) {{
#                 var df_mov = df_filtrado.filter(d => d.Movimiento === mov);
#                 data.push({{
#                     x: df_mov.map(d => d.Fecha_MSE),
#                     y: df_mov.map(d => d.Fecha_SITRA),
#                     mode: 'markers',
#                     marker: {{
#                         size: 8,
#                         opacity: 0.6,
#                         color: i === 0 ? '#4C78A8' : '#F58518'
#                     }},
#                     name: mov,
#                     hovertemplate: '<b>' + mov + '</b><br>MSE: %{{x}}<br>SITRA: %{{y}}<extra></extra>'
#                 }});
#             }});
            
#             // Línea y=x (referencia)
#             if (df_filtrado.length > 0) {{
#                 var minX = Math.min(...df_filtrado.map(d => new Date(d.Fecha_MSE)));
#                 var maxX = Math.max(...df_filtrado.map(d => new Date(d.Fecha_MSE)));
#                 data.push({{
#                     x: [minX, maxX],
#                     y: [minX, maxX],
#                     mode: 'lines',
#                     line: {{color: 'red', dash: 'dash', width: 2}},
#                     name: 'Referencia (y=x)',
#                     hoverinfo: 'skip'
#                 }});
#             }}
            
#             var layout = {{
#                 title: 'Puntos cercanos a la línea roja indican buena sincronización',
#                 xaxis: {{title: 'Fecha MSE'}},
#                 yaxis: {{title: 'Fecha SITRA'}},
#                 hovermode: 'closest',
#                 height: 600
#             }};
            
#             Plotly.newPlot('fig_comparacion', data, layout);
#         }}
#     </script>
# </body>
# </html>
# """
    
#     # Guardar página de estación
#     filename = f"estacion_{codigo}.html"
#     with open(os.path.join(base_path, filename), "w", encoding="utf-8") as f:
#         f.write(html_estacion)

# print(f"✅ Dashboard creado exitosamente en: {base_path}")
# print(f"📁 Archivos generados:")
# print(f"   - index.html (página principal)")
# print(f"   - {len(stats_por_estacion)} páginas de estaciones")
# print(f"\n🌐 Abre 'index.html' en tu navegador para comenzar")

In [ ]:
import pandas as pd
import json
import os

# --- Preparar datos ---
mapeado_filtrado['Retraso en (Segundo)'] = pd.to_numeric(
    mapeado_filtrado['Retraso en (Segundo)'], errors='coerce'
)
df_con_retraso = mapeado_filtrado[mapeado_filtrado['Retraso en (Segundo)'].notna()]

stats_por_estacion = df_con_retraso.groupby('Nombre').agg({
    'Retraso en (Segundo)': 'mean',
    'Código': 'first'
}).reset_index()
stats_por_estacion.columns = ['Estacion', 'Retraso_Promedio', 'Codigo']
stats_por_estacion = stats_por_estacion.sort_values('Retraso_Promedio', ascending=True)

# --- Convertir DataFrame a JSON (incluir todos los datos) ---
df_json = mapeado_filtrado.to_json(orient='records', date_format='iso')

# --- Ruta de guardado ---
base_path = r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\dashboard_retraso_estaciones_1"
os.makedirs(base_path, exist_ok=True)

# --- Crear HTML único con todo el contenido ---
html_completo = f"""
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Dashboard - Retraso por Estación</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
            background-color: #f5f5f5;
        }}
        h1 {{
            color: #333;
            text-align: center;
        }}
        .container {{
            background: white;
            padding: 20px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            margin-bottom: 20px;
        }}
        .info {{
            background: #e3f2fd;
            padding: 10px;
            border-radius: 4px;
            margin-bottom: 20px;
            text-align: center;
        }}
        .btn-back {{
            background: #2196F3;
            color: white;
            padding: 10px 20px;
            border: none;
            border-radius: 4px;
            cursor: pointer;
            text-decoration: none;
            display: inline-block;
        }}
        .btn-back:hover {{
            background: #1976D2;
        }}
        .stats {{
            background: #f0f0f0;
            padding: 15px;
            border-radius: 4px;
            margin-bottom: 20px;
        }}
        .header {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 20px;
        }}
        .vista {{
            display: none;
        }}
        .vista.activa {{
            display: block;
        }}
    </style>
</head>
<body>
    <!-- Vista Principal: Resumen -->
    <div id="vista-resumen" class="vista activa">
        <div class="container">
            <h1>📊 Retraso Promedio por Dependencia <br>(Tiempo MSE - Tiempo Sitra)</h1>
            <div class="info">
                💡 <strong>Haz clic en cualquier barra</strong> para ver el análisis detallado de esa estación
            </div>
            <div id="fig_barras"></div>
        </div>
    </div>

    <!-- Vista Detalle: Estación Individual -->
    <div id="vista-detalle" class="vista">
        <div class="container">
            <div class="header">
                <h1 id="titulo-estacion">🚉</h1>
                <button onclick="volverResumen()" class="btn-back">← Volver al Resumen</button>
            </div>
            
            <div class="stats">
                <strong>Retraso Promedio:</strong> <span id="retraso-estacion"></span> segundos
            </div>
            
            <h2 style="text-align: center;">Comparación MSE vs SITRA</h2>
            <div id="fig_comparacion"></div>
        </div>
    </div>

    <script>
        // Datos globales
        var stats = {json.dumps(stats_por_estacion.to_dict(orient='records'))};
        var df = {df_json};

        // Función para mostrar vista resumen
        function volverResumen() {{
            document.getElementById('vista-resumen').classList.add('activa');
            document.getElementById('vista-detalle').classList.remove('activa');
            window.scrollTo(0, 0);
        }}

        // Función para mostrar vista detalle
        function mostrarDetalle(codigo) {{
            var estacion = stats.find(s => s.Codigo === codigo);
            if (!estacion) return;

            // Actualizar información de la estación
            document.getElementById('titulo-estacion').textContent = '🚉 ' + codigo + ' - ' + estacion.Estacion;
            document.getElementById('retraso-estacion').textContent = estacion.Retraso_Promedio.toFixed(2);

            // Cambiar vista
            document.getElementById('vista-resumen').classList.remove('activa');
            document.getElementById('vista-detalle').classList.add('activa');
            window.scrollTo(0, 0);

            // Generar gráfico de comparación
            generarGraficoComparacion(codigo);
        }}

        // Función para generar gráfico de comparación
        function generarGraficoComparacion(codigo) {{
            var df_filtrado = df.filter(d => d.Código === codigo);
            
            if (df_filtrado.length === 0) {{
                document.getElementById('fig_comparacion').innerHTML = 
                    '<p style="text-align:center;color:#999;">No hay datos disponibles para esta estación</p>';
                return;
            }}

            var data = [];
            var sin_sitra_agregado = false;
            
            ['LLEGADA', 'SALIDA'].forEach(function(mov, i) {{
                var df_mov = df_filtrado.filter(d => d.Movimiento === mov);
                
                // Separar datos con y sin SITRA
                var con_sitra = df_mov.filter(d => d.Fecha_SITRA != null);
                var sin_sitra = df_mov.filter(d => d.Fecha_SITRA == null);
                
                // Puntos normales (con SITRA)
                if (con_sitra.length > 0) {{
                    data.push({{
                        x: con_sitra.map(d => d.Fecha_MSE),
                        y: con_sitra.map(d => d.Fecha_SITRA),
                        mode: 'markers',
                        marker: {{
                            size: 8,
                            symbol: i === 0 ? 'circle' : 'square',
                            opacity: 0.9,
                            color: i === 0 ? '#1565C0' : '#EF6C00',
                            line: {{
                                width: 1,
                                color: '#333'
                            }}
                        }},
                        name: mov,
                        text: con_sitra.map(d => d.NTécnico || 'N/A'),
                        hovertemplate: '<b>' + mov + '</b><br>NTécnico: %{{text}}<br>MSE: %{{x}}<br>SITRA: %{{y}}<extra></extra>'
                    }});
                }}
                
                // Cruces negras (sin SITRA)
                if (sin_sitra.length > 0) {{
                    data.push({{
                        x: sin_sitra.map(d => d.Fecha_MSE),
                        y: sin_sitra.map(d => d.Fecha_MSE),
                        mode: 'markers',
                        marker: {{
                            size: 10,
                            symbol: 'x',
                            color: 'black'
                        }},
                        name: sin_sitra_agregado ? '' : 'Sin SITRA',
                        showlegend: !sin_sitra_agregado,
                        text: sin_sitra.map(d => d.NTécnico || 'N/A'),
                        hovertemplate: '<b>' + mov + ' - SIN DATO SITRA</b><br>NTécnico: %{{text}}<br>MSE: %{{x}}<extra></extra>'
                    }});
                    sin_sitra_agregado = true;
                }}
            }});
            
            // Línea y=x (referencia)
            if (df_filtrado.length > 0) {{
                var fechas_mse = df_filtrado.map(d => new Date(d.Fecha_MSE));
                var minX = Math.min(...fechas_mse);
                var maxX = Math.max(...fechas_mse);
                data.push({{
                    x: [minX, maxX],
                    y: [minX, maxX],
                    mode: 'lines',
                    line: {{color: 'red', dash: 'dash', width: 2}},
                    name: 'Referencia (y=x)',
                    hoverinfo: 'skip'
                }});
            }}

            var layout = {{
                title: 'Los puntos que se encuentran por encima de la línea roja indican que el MSE se encuentra adelantado con respecto a Sitra',
                xaxis: {{title: 'Fecha MSE'}},
                yaxis: {{title: 'Fecha SITRA'}},
                hovermode: 'closest',
                height: 600
            }};
            
            Plotly.newPlot('fig_comparacion', data, layout);
        }}

        // Inicializar gráfico de barras
        function inicializarGraficoBarras() {{
            var barra = {{
                y: stats.map(r => r.Codigo + " - " + r.Estacion),
                x: stats.map(r => r.Retraso_Promedio),
                type: 'bar',
                orientation: 'h',
                marker: {{
                    color: stats.map(r => r.Retraso_Promedio > 0 ? '#4CAF50' : '#f44336')
                }},
                text: stats.map(r => r.Retraso_Promedio.toFixed(2) + "s"),
                textposition: 'outside',
                hovertemplate: '<b>%{{y}}</b><br>Retraso Promedio: %{{x:.2f}} seg<br><i>Click para ver detalles</i><extra></extra>'
            }};

            var layout_barras = {{
                title: '',
                xaxis: {{title: 'Retraso Promedio (segundos)'}},
                yaxis: {{title: 'Estación', automargin: true}},
                height: Math.max(400, stats.length * 30),
                margin: {{l: 200}}
            }};

            Plotly.newPlot('fig_barras', [barra], layout_barras);

            // Evento click para mostrar detalle
            document.getElementById('fig_barras').on('plotly_click', function(data) {{
                var codigo = data.points[0].y.split(' - ')[0];
                mostrarDetalle(codigo);
            }});
        }}

        // Inicializar al cargar la página
        inicializarGraficoBarras();
    </script>
</body>
</html>
"""

# Guardar archivo único
with open(os.path.join(base_path, "dashboard_completo.html"), "w", encoding="utf-8") as f:
    f.write(html_completo)

print(f"✅ Dashboard creado exitosamente en: {base_path}")
print(f"📁 Archivo generado: dashboard_completo.html")
print(f"\n🌐 Abre 'dashboard_completo.html' en tu navegador para comenzar")

In [ ]:
import pandas as pd
import json
import os

# --- Preparar datos ---
mapeado_filtrado['Retraso en (Segundo)'] = pd.to_numeric(
    mapeado_filtrado['Retraso en (Segundo)'], errors='coerce'
)
df_con_retraso = mapeado_filtrado[mapeado_filtrado['Retraso en (Segundo)'].notna()]

stats_por_estacion = df_con_retraso.groupby('Nombre').agg({
    'Retraso en (Segundo)': 'mean',
    'Código': 'first'
}).reset_index()
stats_por_estacion.columns = ['Estacion', 'Retraso_Promedio', 'Codigo']
stats_por_estacion = stats_por_estacion.sort_values('Retraso_Promedio', ascending=True)

# --- Convertir DataFrame a JSON (incluir todos los datos) ---
df_json = mapeado_filtrado.to_json(orient='records', date_format='iso')

# --- Ruta de guardado ---
base_path = r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\dashboard_retraso_estaciones_5"
os.makedirs(base_path, exist_ok=True)

# --- Crear HTML único con todo el contenido ---
html_completo = f"""
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Dashboard - Retraso por Estación</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
            background-color: #f5f5f5;
        }}
        h1 {{
            color: #333;
            text-align: center;
        }}
        .container {{
            background: white;
            padding: 20px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            margin-bottom: 20px;
        }}
        .info {{
            background: #e3f2fd;
            padding: 10px;
            border-radius: 4px;
            margin-bottom: 20px;
            text-align: center;
        }}
        .btn-back {{
            background: #2196F3;
            color: white;
            padding: 10px 20px;
            border: none;
            border-radius: 4px;
            cursor: pointer;
            text-decoration: none;
            display: inline-block;
        }}
        .btn-back:hover {{
            background: #1976D2;
        }}
        .stats {{
            background: #f0f0f0;
            padding: 15px;
            border-radius: 4px;
            margin-bottom: 20px;
        }}
        .header {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 20px;
        }}
        .vista {{
            display: none;
        }}
        .vista.activa {{
            display: block;
        }}
    </style>
</head>
<body>
    <!-- Vista Principal: Resumen -->
    <div id="vista-resumen" class="vista activa">
        <div class="container">
            <h1>📊 Retraso Promedio por Dependencia <br>(Tiempo MSE - Tiempo Sitra)</h1>
            <div class="info">
                💡 <strong>Haz clic en cualquier barra</strong> para ver el análisis detallado de esa estación
            </div>
            <div id="fig_barras"></div>
        </div>
    </div>

    <!-- Vista Detalle: Estación Individual -->
    <div id="vista-detalle" class="vista">
        <div class="container">
            <div class="header">
                <h1 id="titulo-estacion">🚉</h1>
                <button onclick="volverResumen()" class="btn-back">← Volver al Resumen</button>
            </div>
            
            <div class="stats">
                <strong>Retraso Promedio:</strong> <span id="retraso-estacion"></span> segundos
            </div>
            
            <h2 style="text-align: center;">Comparación MSE vs SITRA</h2>
            <div id="fig_comparacion"></div>
        </div>
    </div>

    <script>
        // Datos globales
        var stats = {json.dumps(stats_por_estacion.to_dict(orient='records'))};
        var df = {df_json};

        // Función para mostrar vista resumen
        function volverResumen() {{
            document.getElementById('vista-resumen').classList.add('activa');
            document.getElementById('vista-detalle').classList.remove('activa');
            window.scrollTo(0, 0);
        }}

        // Función para mostrar vista detalle
        function mostrarDetalle(codigo) {{
            var estacion = stats.find(s => s.Codigo === codigo);
            if (!estacion) return;

            // Actualizar información de la estación
            document.getElementById('titulo-estacion').textContent = '🚉 ' + codigo + ' - ' + estacion.Estacion;
            document.getElementById('retraso-estacion').textContent = estacion.Retraso_Promedio.toFixed(2);

            // Cambiar vista
            document.getElementById('vista-resumen').classList.remove('activa');
            document.getElementById('vista-detalle').classList.add('activa');
            window.scrollTo(0, 0);

            // Generar gráfico de comparación
            generarGraficoComparacion(codigo);
        }}

        // Función para generar gráfico de comparación
        function generarGraficoComparacion(codigo) {{
            var df_filtrado = df.filter(d => d.Código === codigo);
            
            if (df_filtrado.length === 0) {{
                document.getElementById('fig_comparacion').innerHTML = 
                    '<p style="text-align:center;color:#999;">No hay datos disponibles para esta estación</p>';
                return;
            }}

            var data = [];
            var sin_sitra_agregado = false;
            
            ['LLEGADA', 'SALIDA'].forEach(function(mov, i) {{
                var df_mov = df_filtrado.filter(d => d.Movimiento === mov);
                
                // Separar datos con y sin SITRA
                var con_sitra = df_mov.filter(d => d.Fecha_SITRA != null);
                var sin_sitra = df_mov.filter(d => d.Fecha_SITRA == null);
                
                // Puntos normales (con SITRA)
                if (con_sitra.length > 0) {{
                    data.push({{
                        x: con_sitra.map(d => d.Fecha_MSE),
                        y: con_sitra.map(d => d.Fecha_SITRA),
                        mode: 'markers',
                        marker: {{
                            size: 8,
                            symbol: i === 0 ? 'circle' : 'square',
                            opacity: 0.9,
                            color: i === 0 ? '#1565C0' : '#EF6C00',
                            line: {{
                                width: 1,
                                color: '#333'
                            }}
                        }},
                        name: mov,
                        text: con_sitra.map(d => d.NTécnico || 'N/A'),
                        hovertemplate: '<b>' + mov + '</b><br>NTécnico: %{{text}}<br>MSE: %{{x}}<br>SITRA: %{{y}}<extra></extra>'
                    }});
                }}
                
                // Cruces negras (sin SITRA)
                if (sin_sitra.length > 0) {{
                    data.push({{
                        x: sin_sitra.map(d => d.Fecha_MSE),
                        y: sin_sitra.map(d => d.Fecha_MSE),
                        mode: 'markers',
                        marker: {{
                            size: 10,
                            symbol: 'x',
                            color: 'black'
                        }},
                        name: sin_sitra_agregado ? '' : 'Sin SITRA',
                        showlegend: !sin_sitra_agregado,
                        text: sin_sitra.map(d => d.NTécnico || 'N/A'),
                        hovertemplate: '<b>' + mov + ' - SIN DATO SITRA</b><br>NTécnico: %{{text}}<br>MSE: %{{x}}<extra></extra>'
                    }});
                    sin_sitra_agregado = true;
                }}
            }});
            
            // Línea y=x (referencia)
            if (df_filtrado.length > 0) {{
                var fechas_mse = df_filtrado.map(d => new Date(d.Fecha_MSE));
                var minX = Math.min(...fechas_mse);
                var maxX = Math.max(...fechas_mse);
                data.push({{
                    x: [minX, maxX],
                    y: [minX, maxX],
                    mode: 'lines',
                    line: {{color: 'red', dash: 'dash', width: 2}},
                    name: 'Referencia (y=x)',
                    hoverinfo: 'skip'
                }});
            }}

            var layout = {{
                title: 'Los puntos que se encuentran por encima de la línea roja indican que el MSE se encuentra adelantado con respecto a Sitra',
                xaxis: {{title: 'Fecha MSE'}},
                yaxis: {{title: 'Fecha SITRA'}},
                hovermode: 'closest',
                height: 600
            }};
            
            Plotly.newPlot('fig_comparacion', data, layout);
        }}

        // Inicializar gráfico de barras
        function inicializarGraficoBarras() {{
            var barra = {{
                y: stats.map(r => r.Codigo + " - " + r.Estacion),
                x: stats.map(r => r.Retraso_Promedio),
                type: 'bar',
                orientation: 'h',
                marker: {{
                    color: stats.map(r => r.Retraso_Promedio > 0 ? '#4CAF50' : '#f44336')
                }},
                text: stats.map(r => r.Retraso_Promedio.toFixed(2) + "s"),
                textposition: 'outside',
                hovertemplate: '<b>%{{y}}</b><br>Retraso Promedio: %{{x:.2f}} seg<br><i>Click para ver detalles</i><extra></extra>'
            }};

            var layout_barras = {{
                title: '',
                xaxis: {{title: 'Retraso Promedio (segundos)'}},
                yaxis: {{title: 'Estación', automargin: true}},
                height: Math.max(400, stats.length * 30),
                margin: {{l: 200}}
            }};

            Plotly.newPlot('fig_barras', [barra], layout_barras);

            // Evento click para mostrar detalle
            document.getElementById('fig_barras').on('plotly_click', function(data) {{
                var codigo = data.points[0].y.split(' - ')[0];
                mostrarDetalle(codigo);
            }});
        }}

        // Inicializar al cargar la página
        inicializarGraficoBarras();
    </script>
</body>
</html>
"""

# Guardar archivo único
with open(os.path.join(base_path, "dashboard_completo.html"), "w", encoding="utf-8") as f:
    f.write(html_completo)

print(f"✅ Dashboard creado exitosamente en: {base_path}")
print(f"📁 Archivo generado: dashboard_completo.html")
print(f"\n🌐 Abre 'dashboard_completo.html' en tu navegador para comenzar")

In [ ]:
stats_por_estacion.rename(columns={"Estacion":"Nombre","Retraso_Promedio":"Retraso_Promedio(s)"},inplace=True)

In [ ]:
stats_por_estacion =   stats_por_estacion[["Nombre","Codigo","Retraso_Promedio(s)"]].copy()



In [ ]:
fname =Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\2025_12_17_Comparación_MSE_SITRA.xlsx")
data={
    "Resumen_Retrasos":stats_por_estacion,
    "Detalle_Retrasos":mapeado_filtrado
}
guardarExcelMulti(data,fname)

In [ ]:
import xml.etree.ElementTree as ET
import re
tfile = Path("data/tablas auxiliares/adifControlPointTable_v00.xml")
with tfile.open("r", encoding="utf8") as f:
    track_txt = f.read()
track_info = ET.fromstring(track_txt)


ns_uri = re.match(r'\{(.+)\}', track_info.tag).group(1)
ns = {"ns": ns_uri}

records = []
for cp in track_info.findall("ns:controlPoint", ns):
    row = dict(cp.attrib)

    for child in ["KMPoint", "pointType", "regulationPoint", "shortCode"]:
        el = cp.find(f"ns:{child}", ns)
        if el is not None:
            row[child] = el.get("value")

    for child in ["area", "region"]:
        el = cp.find(f"ns:{child}", ns)
        if el is not None:
            row[child] = el.get("code")

    prov = cp.find("ns:province", ns)
    if prov is not None:
        row["province"] = prov.get("code")
        row["provinceINE"] = prov.get("INECode")

    records.append(row)

df = pd.DataFrame(records)
df

In [ ]:
df["regulationPoint"].unique()

In [ ]:
sitra = df[df["regulationPoint"] =="S"]

In [ ]:
sitra

In [ ]:
coor = Path("data/coordenada.csv")

In [ ]:
coordenadas = pd.read_csv(coor)

In [ ]:
coordenadas_sitra = coordenadas[coordenadas["Código"].isin(sitra["code"])]

In [ ]:
coordenadas_sitra[(coordenadas_sitra["Longitud"].isna()) | (coordenadas_sitra["Latitud"].isna())]

In [ ]:
test= sitra.merge(coordenadas_sitra, left_on="code", right_on="Código", how="left")

In [ ]:
coordenadas_sitra["Latitud"] = (
    coordenadas_sitra["Latitud"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)
coordenadas_sitra["Longitud"] = (
    coordenadas_sitra["Longitud"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)
df_filtrado = coordenadas_sitra[
    coordenadas_sitra["Longitud"].round(2) == coordenadas_sitra["Longitud"]
]

In [ ]:
df_filtrado

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Coordenadas_puntos_SITRA_1.xlsx")

In [ ]:
guardarExcel(coordenadas_sitra, fname)

In [ ]:
coordenadas_sitra


In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\ControlPoint.xlsx")

In [ ]:
guardarExcel(df, fname)

In [ ]:
sitra[sitra["code"] == "11405"]

In [ ]:
estaciones = loadEstaciones()

In [ ]:
duplicado = estaciones[estaciones["Código"].duplicated(keep=False)].copy()

In [ ]:
duplicado.sort_values(by="Código", inplace=True)

In [ ]:
mask = duplicado.groupby('Código')['NombreCTC'].transform(lambda x: x.str.contains('AV').any())

resultado = duplicado[mask]

In [ ]:
fname= Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\estaciones_RC_av.xlsx")

In [ ]:
guardarExcel(resultado,fname)

In [ ]:
df = historico_pro.copy()

In [ ]:
historico_pro["Producto"].unique()


In [ ]:
lista_subdf = [grupo for _, grupo in historico_pro.groupby('NTécnico')]

In [ ]:
resultados = []

for grupo in lista_subdf:
    origenes = grupo[grupo['Movimiento'] == 'ORIGEN']
    fines = grupo[grupo['Movimiento'] == 'FIN']
    
    if not origenes.empty and not fines.empty:
        ultimo_origen = origenes.iloc[-1]['Fecha']
        
        # Buscar el primer FIN posterior al último ORIGEN
        fines_posteriores = fines[fines['Fecha'] > ultimo_origen]
        
        if fines_posteriores.empty:
            continue  # No hay FIN posterior al ORIGEN, saltar
        
        primer_fin = fines_posteriores.iloc[0]['Fecha']
        tiempo = primer_fin - ultimo_origen
        
        # Convertir a hh:mm:ss
        total_segundos = int(tiempo.total_seconds())
        horas, resto = divmod(total_segundos, 3600)
        minutos, segundos = divmod(resto, 60)
        tiempo_formato = f"{horas:02}:{minutos:02}:{segundos:02}"
        
        # Clasificación
        minutos_total = total_segundos / 60
        if minutos_total < 5:
            clasificacion = '< 5 minutos'
        elif minutos_total < 10:
            clasificacion = '< 10 minutos'
        elif minutos_total < 15:
            clasificacion = '< 15 minutos'
        elif minutos_total < 20:
            clasificacion = '< 20 minutos'
        elif minutos_total < 25:
            clasificacion = '< 25 minutos'
        elif minutos_total < 30:
            clasificacion = '< 30 minutos'
        elif minutos_total < 60:
            clasificacion = '< 1 hora'
        elif minutos_total < 120:
            clasificacion = '1-2 horas'
        elif minutos_total < 180:
            clasificacion = '2-3 horas'
        elif minutos_total < 240:
            clasificacion = '3-4 horas'
        elif minutos_total < 300:
            clasificacion = '4-5 horas'
        else:
            clasificacion = '> 5 horas'
        
        resultados.append({
            'NTécnico': grupo.iloc[0]["NTécnico"],
            'Producto': grupo.iloc[0]["Producto"],
            'Origen': ultimo_origen,
            'Fin': primer_fin,
            'tiempo_recorrido': tiempo_formato,
            'Clasificación': clasificacion,
        })

df_resultado = pd.DataFrame(resultados)

In [ ]:
df_resultado

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\tiempo_recorrido.xlsx")
guardarExcel(df_resultado,fname)

In [ ]:
import pandas as pd
import json
import requests
import regex
def hacerPeticion(method: str, URL: str, headers: dict = {}, data: dict = None):

    headers = {
        "Content-Type": "application/json; charset=UTF-8",
        "Prefer": "respond-async",
        **headers,
    }

    response = requests.request(method, URL, headers=headers, data=data)
    if response.status_code == 200:
        return response
    print("")
    if response.status_code >= 400:
        print(f"Error: '{response.status_code}' en la respuesta")
        return
    if response.status_code >= 300:
        print(f"Redirección: código'{response.status_code}'")
        return
    if response.status_code > 200:
        print(f"👍: código'{response.status_code}'")
        return
    if response.status_code < 200:
        print(f"Info: código'{response.status_code}'")
        return
    if not response.encoding == "utf-8":
        response.encoding = "utf-8"
    return response
HOSTPATH = "http://info.api.elcano.operaciones.adif/mse-circulations/msecirculations/planning/day/"
data = {"day": pd.to_datetime("2026-04-09").strftime("%Y-%m-%d")}
data = json.dumps(data)

response = hacerPeticion(
    "POST",
    HOSTPATH,
    data=data,
)
res_data = regex.sub(r"\n*data:\s*", ",", response.text)[1:]
res_data = json.loads(f"[{res_data}]")
# gct = pd.DataFrame(
#     [
#         {
#             "NTécnico": el["circulationId"]["number"],
#             "Fecha": "-".join(f"{i}" for i in el["circulationId"]["launchingDate"]),
#             "NComercial": el["dayTrain"]["commercialNumber"],
#             "Línea": el["dayTrain"]["line"],
#             "Empresa": el["dayTrain"]["company"],
#             "Operador": el["dayTrain"]["operator"],
#             "Tipo": el["dayTrain"]["trainType"],
#             "esComercial": el["dayTrain"]["commercialTrain"],
#             "esEspecial": el["dayTrain"]["special"],
#             "esVirtual": el["dayTrain"]["virtual"],
#             "Recorrido": el["dayTrain"]["journey"],
#             "Rotación": el["dayTrain"]["connections"]["next"]["circulationId"]["number"]
#         }
#         for el in res_data
#     ]
# )
# gct["Línea"] = gct["Línea"].apply(lambda x: x.get("name") if x else x)


In [ ]:
with open("datos.json", "w", encoding="utf-8") as f:
    json.dump(res_data, f, ensure_ascii=False, indent=4)

In [ ]:
res_data

<h1> Apeadero Mismo CV </h1>

In [ ]:
apeaderos = pd.read_csv("data/Apeaderos.csv")

In [ ]:
apeaderos["Código"] = apeaderos["Código"].astype(str)

In [ ]:
apeaderos["Código"] = apeaderos["Código"].apply(rellenarId)

In [ ]:
apeaderos[apeaderos["Código"] =="60600"]

In [ ]:
import re
import os
# Directorio raíz donde buscar archivos .pl
root_dir = Path(r"C:\topogen-adif-repo\baseline")
# Lista para almacenar los resultados
datos = []

# Expresiones regulares
re_nombre = re.compile(r'%% Nombre Estacion: (\d+)-(.+?) %%')  # Captura código y nombre
re_recta_cv = re.compile(r"recta\('(.+?)',\s*'(.+?)',\s*cv_estacionamiento,")  # Solo rectas con cv_estacionamiento

# Recorrer todos los subdirectorios
for dirpath, dirnames, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith('.pl'):
            filepath = os.path.join(dirpath, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
                
                # Buscar el código y nombre de la estación
                match_nombre = re_nombre.search(content)
                if match_nombre:
                    codigo = match_nombre.group(1)
                    nombre = match_nombre.group(2)
                else:
                    codigo = None
                    nombre = None
                
                # Buscar todas las rectas con cv_estacionamiento
                rectas = re_recta_cv.findall(content)
                
                # Agregar cada recta como fila independiente
                for recta, estacion in rectas:
                    datos.append({
                        'Código': codigo,
                        'Nombre': nombre,
                        'Recta Estacionamiento': f"{recta}" 
                    })

# Crear DataFrame
df = pd.DataFrame(datos)

In [ ]:
df["Código"] = df["Código"].astype(str)


In [ ]:
apeaderos_cv = df[df["Código"].isin(apeaderos["Código"])].copy()

In [ ]:
# # Mantener solo la parte antes del guion bajo en la columna 'Recta Estacionamiento'
# apeaderos_cv['Recta Estacionamiento'] = apeaderos_cv['Recta Estacionamiento'].str.split('_').str[-1]

In [ ]:
apeaderos_cv

In [ ]:
duplicado = apeaderos_cv[apeaderos_cv["Recta Estacionamiento"].duplicated(keep=False)].copy()

In [ ]:
duplicado.sort_values(by="Recta Estacionamiento", inplace=True)


In [ ]:
duplicado

In [ ]:
guardarExcel(duplicado, Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\apeaderos_cv_1.xlsx"))

In [ ]:
apeaderos[apeaderos["Código"] == "70005"]

In [ ]:
fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\apeaderos_cv_1.xlsx")

In [ ]:
df = pd.read_excel(fname)

In [ ]:
df["Código"] = df["Código"].apply(rellenarId)

In [ ]:
df

In [ ]:
estaciones = loadEstaciones()

In [ ]:
subdirecciones = pd.read_csv(r"data/Subdirección_2.csv")

In [ ]:
subdirecciones.drop(columns="Nombre", inplace=True)

In [ ]:
merged =pd.merge(df, subdirecciones, left_on="Código", right_on="Código", how="left")

In [ ]:
merged

In [ ]:
estaciones = estaciones[["CTC","Código"]].copy()

In [ ]:
merged =pd.merge(merged, estaciones, left_on="Código", right_on="Código", how="left")

In [ ]:
merged

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\apeaderos_cv_3.xlsx")

In [ ]:
guardarExcel(merged,fname)

In [ ]:
fname = Path(r"c:\Users\xiangzhou.zhang\OneDrive - Ingeniería y Economía del Transporte S.A\Backlog\Data\Informe_puntual\estaciones_sin_abreviatura.txt")

In [ ]:
estacion = loadEstaciones()

In [ ]:
with open(fname, "r", encoding="utf-8") as f:
    estaciones = [line.strip() for line in f if line.strip()]

In [ ]:
estacion[estacion["Código"] == "05534"]

In [ ]:
estaciones_sin_ctc = loadEstacionSinCTC()

In [ ]:
estaciones_sin_ctc[estaciones_sin_ctc["Código"] == "A0660"]

In [ ]:
elementos_presentes = [e for e in estaciones if e in estacion['Nombre'].values]
elementos_ausentes  = [e for e in estaciones if e not in estacion['Nombre'].values]

In [ ]:
import re

def extraer_datos_sql(ruta_archivo):
    with open(ruta_archivo, "r", encoding="utf-8") as f:
        sql = f.read()

    pattern = r"ABREVIATURA = (?:'(\w+)'|(null)) WHERE id_punto_regulacion = '(\w+)'"
    
    data = []
    for abr, null, id_ in re.findall(pattern, sql, re.IGNORECASE):
        data.append({
            "id_punto_regulacion": id_,
            "abreviatura": abr if abr else None
        })

    return data

data = extraer_datos_sql(fname)

In [ ]:
estacion["Mnemónico"] = estacion["Mnemónico"].where(estacion["Mnemónico"].notna(), None)

In [ ]:
def verificar_nemonico(df, datos_sql):
    # Convertir los datos SQL a DataFrame
    df_sql = pd.DataFrame(datos_sql)

    # Merge entre el DataFrame original y los datos SQL
    df_merged = df_sql.merge(
        df[["Código", "Mnemónico"]],
        left_on="id_punto_regulacion",
        right_on="Código",
        how="left"
    )

    # Comparar abreviatura con nemónico (ambos None también cuenta como igual)
    df_merged["coincide"] = df_merged["abreviatura"] == df_merged["Mnemónico"]

    return df_merged[["id_punto_regulacion", "abreviatura", "Mnemónico", "coincide"]]

In [ ]:
resultado  = verificar_nemonico(estacion, data)

In [ ]:
false = resultado[resultado["coincide"] == False]

In [ ]:
false

In [ ]:
false_1 = false[(false["abreviatura"] != None) & (~false["Mnemónico"].isna())]


In [ ]:
fname= Path(r"C:\Users\xiangzhou.zhang\OneDrive - Ingeniería y Economía del Transporte S.A\Backlog\Data\Informe_puntual\Comparación_nemónicos.xlsx")

In [ ]:
guardarExcel(false_1, fname)

In [ ]:
estacion[estacion["Código"] == "05534"]